In [ ]:
# Mapeia diretório de uso
from google.colab import drive
import os
import pandas as pd
import numpy as np

# Se você estiver usando o Google Colab, descomente as duas linhas abaixo
drive.mount('/content/drive', force_remount=True)
os.chdir('/content/drive/MyDrive/TCC_2/dados')

In [ ]:
# ============================================================
# AUDITORIA RAIS PROFESSORES
# PARTE 1 - ARQUIVOS PARQUET JÁ FILTRADOS
#
# Verifica:
#   - quantidade por ano e UF
#   - quantidade por família CBO
#   - quantidade por CBO de 6 dígitos
#   - distribuição percentual das famílias
#   - variação entre anos
#   - UFs incompatíveis com o arquivo regional
#   - valores ausentes de UF
#
# NÃO baixa arquivos novamente.
# ============================================================

from google.colab import drive

import os
import re
import glob
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

from collections import Counter


# ============================================================
# 1. GOOGLE DRIVE
# ============================================================

drive.mount(
    '/content/drive',
    force_remount=False
)


# ============================================================
# 2. CONFIGURAÇÕES
# ============================================================

PASTA_PARQUET = (
    '/content/drive/MyDrive/TCC_2/dados/RAIS_PROFESSORES'
)

PASTA_SAIDA = (
    '/content/drive/MyDrive/TCC_2/resultados/'
    'AUDITORIA_RAIS_PROFESSORES'
)

os.makedirs(
    PASTA_SAIDA,
    exist_ok=True
)


ANOS = [
    2020,
    2021,
    2023,
    2024,
    2025
]


FAMILIAS_CBO = {

    '2312':
        'Ensino Fundamental - anos iniciais',

    '2313':
        'Ensino Fundamental - anos finais',

    '2321':
        'Ensino Médio'
}


UFS_ESPERADAS = {

    'NORTE': {
        'AC', 'AP', 'AM', 'PA',
        'RO', 'RR', 'TO'
    },

    'NORDESTE': {
        'AL', 'BA', 'CE', 'MA',
        'PB', 'PE', 'PI', 'RN', 'SE'
    },

    'CENTRO_OESTE': {
        'DF', 'GO', 'MT', 'MS'
    },

    'MG_ES_RJ': {
        'MG', 'ES', 'RJ'
    },

    'SP': {
        'SP'
    },

    'SUL': {
        'PR', 'SC', 'RS'
    }
}


# ============================================================
# 3. LOCALIZAR TODOS OS PARQUETS
# ============================================================

arquivos_parquet = sorted(
    glob.glob(
        os.path.join(
            PASTA_PARQUET,
            '**',
            '*.parquet'
        ),
        recursive=True
    )
)


print(
    f'Arquivos encontrados: '
    f'{len(arquivos_parquet)}'
)


for arquivo in arquivos_parquet:

    print(
        os.path.relpath(
            arquivo,
            PASTA_PARQUET
        )
    )


if len(arquivos_parquet) != 30:

    print(
        '\nATENÇÃO: esperávamos 30 arquivos '
        '(5 anos x 6 grupos).'
    )


# ============================================================
# 4. FUNÇÃO PARA IDENTIFICAR ANO E GRUPO
# ============================================================

def identificar_ano_grupo(caminho):

    nome = os.path.basename(
        caminho
    )

    padrao = (
        r'RAIS_PROFESSORES_'
        r'(\d{4})_(.+)\.parquet$'
    )

    resultado = re.search(
        padrao,
        nome,
        flags=re.IGNORECASE
    )

    if resultado is None:

        raise RuntimeError(
            f'Não consegui interpretar: {nome}'
        )

    ano = int(
        resultado.group(1)
    )

    grupo = (
        resultado
        .group(2)
        .upper()
    )

    return ano, grupo


# ============================================================
# 5. CONTADORES
# ============================================================

contagem_uf = Counter()

contagem_uf_familia = Counter()

contagem_uf_cbo = Counter()

validacao_arquivo = []

resumo_arquivo = []


# ============================================================
# 6. LER PARQUETS EM BLOCOS
# ============================================================

for i, arquivo in enumerate(
    arquivos_parquet,
    start=1
):

    ano, grupo = (
        identificar_ano_grupo(
            arquivo
        )
    )


    print('\n' + '=' * 80)

    print(
        f'[{i}/{len(arquivos_parquet)}] '
        f'{ano} - {grupo}'
    )

    print('=' * 80)


    parquet = pq.ParquetFile(
        arquivo
    )


    colunas = (
        parquet.schema.names
    )


    # --------------------------------------------------------
    # Identificar as colunas que criamos
    # --------------------------------------------------------

    if 'UF' not in colunas:

        raise RuntimeError(
            f'Coluna UF ausente em {arquivo}'
        )


    if 'Familia_CBO' not in colunas:

        raise RuntimeError(
            f'Coluna Familia_CBO ausente em {arquivo}'
        )


    if 'CBO_padronizada' not in colunas:

        raise RuntimeError(
            f'Coluna CBO_padronizada ausente em {arquivo}'
        )


    total_arquivo = 0

    uf_nula = 0

    ufs_arquivo = Counter()


    # --------------------------------------------------------
    # Ler SOMENTE as 3 colunas necessárias
    # --------------------------------------------------------

    for batch in parquet.iter_batches(

        batch_size=250_000,

        columns=[
            'UF',
            'Familia_CBO',
            'CBO_padronizada'
        ]
    ):

        df = batch.to_pandas()


        total_arquivo += len(
            df
        )


        # ----------------------------------------------------
        # Padronizar
        # ----------------------------------------------------

        df['UF'] = (
            df['UF']
            .astype('string')
            .str.strip()
            .str.upper()
        )


        df['Familia_CBO'] = (
            df['Familia_CBO']
            .astype('string')
            .str.extract(
                r'(\d{4})',
                expand=False
            )
        )


        df['CBO_padronizada'] = (
            df['CBO_padronizada']
            .astype('string')
            .str.extract(
                r'(\d{6})',
                expand=False
            )
        )


        # ----------------------------------------------------
        # UFs nulas
        # ----------------------------------------------------

        mascara_uf_nula = (
            df['UF'].isna()
            |
            (df['UF'] == '')
        )


        uf_nula += (
            mascara_uf_nula.sum()
        )


        df_validos = df.loc[
            ~mascara_uf_nula
        ].copy()


        # ----------------------------------------------------
        # Contagem UF
        # ----------------------------------------------------

        contagem_batch = (
            df_validos
            .groupby(
                'UF',
                dropna=False
            )
            .size()
        )


        for uf, qtd in (
            contagem_batch.items()
        ):

            contagem_uf[
                (
                    ano,
                    uf
                )
            ] += int(qtd)

            ufs_arquivo[
                uf
            ] += int(qtd)


        # ----------------------------------------------------
        # UF x Família
        # ----------------------------------------------------

        contagem_batch = (
            df_validos
            .groupby(
                [
                    'UF',
                    'Familia_CBO'
                ],
                dropna=False
            )
            .size()
        )


        for (
            uf,
            familia
        ), qtd in contagem_batch.items():

            contagem_uf_familia[
                (
                    ano,
                    uf,
                    familia
                )
            ] += int(qtd)


        # ----------------------------------------------------
        # UF x CBO de 6 dígitos
        # ----------------------------------------------------

        contagem_batch = (
            df_validos
            .groupby(
                [
                    'UF',
                    'CBO_padronizada'
                ],
                dropna=False
            )
            .size()
        )


        for (
            uf,
            cbo
        ), qtd in contagem_batch.items():

            contagem_uf_cbo[
                (
                    ano,
                    uf,
                    cbo
                )
            ] += int(qtd)


    # --------------------------------------------------------
    # Validar UF contra o arquivo regional
    # --------------------------------------------------------

    encontradas = set(
        ufs_arquivo.keys()
    )


    esperadas = (
        UFS_ESPERADAS.get(
            grupo,
            set()
        )
    )


    invalidas = (
        encontradas
        -
        esperadas
    )


    ausentes = (
        esperadas
        -
        encontradas
    )


    validacao_arquivo.append({

        'Ano':
            ano,

        'Grupo':
            grupo,

        'Total_registros':
            total_arquivo,

        'UF_nula':
            uf_nula,

        'Percentual_UF_nula':
            (
                uf_nula
                / total_arquivo
                * 100
                if total_arquivo > 0
                else np.nan
            ),

        'UFs_encontradas':
            ', '.join(
                sorted(
                    encontradas
                )
            ),

        'UFs_invalidas':
            ', '.join(
                sorted(
                    invalidas
                )
            ),

        'UFs_esperadas_ausentes':
            ', '.join(
                sorted(
                    ausentes
                )
            )
    })


    resumo_arquivo.append({

        'Ano':
            ano,

        'Grupo':
            grupo,

        'Professores':
            total_arquivo
    })


    print(
        f'Registros: '
        f'{total_arquivo:,}'
    )

    print(
        f'UF nula: '
        f'{uf_nula:,}'
    )

    print(
        'UFs encontradas: '
        f'{sorted(encontradas)}'
    )


    if len(invalidas) > 0:

        print(
            'ATENÇÃO - UFs inválidas: '
            f'{sorted(invalidas)}'
        )


# ============================================================
# 7. DATAFRAME - ANO x UF
# ============================================================

dados = []


for (
    ano,
    uf
), qtd in contagem_uf.items():

    dados.append({

        'Ano':
            ano,

        'UF':
            uf,

        'Professores':
            qtd
    })


df_uf = pd.DataFrame(
    dados
)


df_uf = (
    df_uf
    .sort_values(
        [
            'UF',
            'Ano'
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 8. PARTICIPAÇÃO NACIONAL DE CADA UF
# ============================================================

df_uf[
    'Percentual_Brasil'
] = (

    df_uf['Professores']

    /

    df_uf
    .groupby(
        'Ano'
    )[
        'Professores'
    ]
    .transform(
        'sum'
    )

    * 100
)


# ============================================================
# 9. VARIAÇÃO ENTRE ANOS DISPONÍVEIS
# ============================================================

df_variacao = (
    df_uf
    .copy()
)


df_variacao[
    'Ano_anterior'
] = (

    df_variacao
    .groupby(
        'UF'
    )[
        'Ano'
    ]
    .shift(1)
)


df_variacao[
    'Professores_ano_anterior'
] = (

    df_variacao
    .groupby(
        'UF'
    )[
        'Professores'
    ]
    .shift(1)
)


df_variacao[
    'Intervalo_anos'
] = (

    df_variacao['Ano']
    -
    df_variacao['Ano_anterior']
)


df_variacao[
    'Variacao_total_pct'
] = (

    (
        df_variacao['Professores']
        /
        df_variacao[
            'Professores_ano_anterior'
        ]
        -
        1
    )

    * 100
)


# ------------------------------------------------------------
# Variação anualizada:
# importante porque 2021 -> 2023 tem intervalo de 2 anos
# ------------------------------------------------------------

df_variacao[
    'Variacao_anualizada_pct'
] = (

    (
        (
            df_variacao['Professores']
            /
            df_variacao[
                'Professores_ano_anterior'
            ]
        )
        **
        (
            1
            /
            df_variacao[
                'Intervalo_anos'
            ]
        )
        -
        1
    )

    * 100
)


# ------------------------------------------------------------
# Sinalizar oscilações altas
#
# Aqui usamos 15% ao ano SOMENTE como alerta para auditoria.
# Não significa automaticamente que o dado esteja errado.
# ------------------------------------------------------------

df_variacao[
    'Alerta_variacao'
] = (

    df_variacao[
        'Variacao_anualizada_pct'
    ]
    .abs()
    >
    15
)


# ============================================================
# 10. ANO x UF x FAMÍLIA CBO
# ============================================================

dados = []


for (
    ano,
    uf,
    familia
), qtd in contagem_uf_familia.items():

    dados.append({

        'Ano':
            ano,

        'UF':
            uf,

        'Familia_CBO':
            familia,

        'Descricao':
            FAMILIAS_CBO.get(
                str(familia),
                'Outra / não identificada'
            ),

        'Professores':
            qtd
    })


df_uf_familia = pd.DataFrame(
    dados
)


df_uf_familia[
    'Percentual_na_UF'
] = (

    df_uf_familia[
        'Professores'
    ]

    /

    df_uf_familia
    .groupby(
        [
            'Ano',
            'UF'
        ]
    )[
        'Professores'
    ]
    .transform(
        'sum'
    )

    * 100
)


df_uf_familia = (

    df_uf_familia
    .sort_values(
        [
            'Ano',
            'UF',
            'Familia_CBO'
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 11. CONTAGEM NACIONAL POR FAMÍLIA
# ============================================================

df_familia_nacional = (

    df_uf_familia

    .groupby(
        [
            'Ano',
            'Familia_CBO',
            'Descricao'
        ],
        as_index=False
    )

    ['Professores']

    .sum()
)


df_familia_nacional[
    'Percentual'
] = (

    df_familia_nacional[
        'Professores'
    ]

    /

    df_familia_nacional
    .groupby(
        'Ano'
    )[
        'Professores'
    ]
    .transform(
        'sum'
    )

    * 100
)


# ============================================================
# 12. ANO x UF x CBO 6 DÍGITOS
# ============================================================

dados = []


for (
    ano,
    uf,
    cbo
), qtd in contagem_uf_cbo.items():

    dados.append({

        'Ano':
            ano,

        'UF':
            uf,

        'CBO_6digitos':
            cbo,

        'Familia_CBO':
            (
                str(cbo)[:4]
                if pd.notna(cbo)
                else None
            ),

        'Professores':
            qtd
    })


df_cbo = pd.DataFrame(
    dados
)


df_cbo = (

    df_cbo
    .sort_values(
        [
            'Ano',
            'UF',
            'Professores'
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 13. VALIDAÇÃO DOS ARQUIVOS
# ============================================================

df_validacao = pd.DataFrame(
    validacao_arquivo
)


df_resumo_arquivo = pd.DataFrame(
    resumo_arquivo
)


# ============================================================
# 14. RESUMO NACIONAL
# ============================================================

df_nacional = (

    df_uf

    .groupby(
        'Ano',
        as_index=False
    )

    ['Professores']

    .sum()
)


# ============================================================
# 15. SALVAR RESULTADOS
# ============================================================

df_nacional.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '01_professores_por_ano_brasil.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_uf.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '02_professores_por_ano_uf.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_variacao.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '03_variacao_professores_por_uf.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_uf_familia.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '04_professores_uf_familia_cbo.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_familia_nacional.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '05_familias_cbo_brasil.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_cbo.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '06_cbo_6digitos_por_uf.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_validacao.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '07_validacao_uf_grupos.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 16. EXIBIR RESULTADOS PRINCIPAIS
# ============================================================

print('\n' + '=' * 80)
print('TOTAL BRASIL')
print('=' * 80)

display(
    df_nacional
)


print('\n' + '=' * 80)
print('PROFESSORES POR ANO E UF')
print('=' * 80)

display(
    df_uf
)


print('\n' + '=' * 80)
print('DISTRIBUIÇÃO NACIONAL POR FAMÍLIA CBO')
print('=' * 80)

display(
    df_familia_nacional
)


print('\n' + '=' * 80)
print('ALERTAS DE VARIAÇÃO')
print('=' * 80)

display(

    df_variacao.loc[
        df_variacao[
            'Alerta_variacao'
        ]
        ==
        True
    ]
    .sort_values(
        'Variacao_anualizada_pct',
        key=abs,
        ascending=False
    )
)


print('\n' + '=' * 80)
print('VALIDAÇÃO DAS UFs')
print('=' * 80)

display(
    df_validacao
)


print(
    '\nArquivos salvos em:\n'
    f'{PASTA_SAIDA}'
)

In [ ]:
# ============================================================
# AUDITORIA RAIS PROFESSORES
# PARTE 2 - VERIFICAR FAMÍLIAS CBO OMITIDAS
#
# Famílias analisadas:
#
# 2312 = professores nível superior - fundamental anos iniciais
# 2313 = professores nível superior - fundamental anos finais
# 2321 = professores do ensino médio
#
# 3312 = professores de nível médio no ensino fundamental
# 3321 = professores leigos no ensino fundamental
#
# IMPORTANTE:
# - NÃO FAZ DOWNLOAD
# - utiliza os .7z que já estão no Google Drive
# - extrai um arquivo por vez no disco temporário do Colab
# - lê somente CBO e Município
# - apaga o arquivo bruto ao terminar
# ============================================================

import os
import re
import glob
import shutil
import subprocess
import unicodedata

import pandas as pd
import numpy as np

from collections import Counter


# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

PASTA_BASE = (
    '/content/drive/MyDrive/TCC_2/dados/RAIS_OUTROS_ESTADOS'
)

PASTA_SAIDA = (
    '/content/drive/MyDrive/TCC_2/resultados/'
    'AUDITORIA_RAIS_PROFESSORES'
)

PASTA_TEMP = (
    '/content/AUDITORIA_RAIS_TEMP'
)


ANOS = [
    2020,
    2021,
    2023,
    2024,
    2025
]


GRUPOS = [
    'NORTE',
    'NORDESTE',
    'CENTRO_OESTE',
    'MG_ES_RJ',
    'SP',
    'SUL'
]


FAMILIAS = {

    '2312':
        'Superior - Fundamental anos iniciais',

    '2313':
        'Superior - Fundamental anos finais',

    '2321':
        'Ensino Médio',

    '3312':
        'Nível médio - Ensino Fundamental',

    '3321':
        'Professor leigo - Ensino Fundamental'
}


MAPA_UF = {

    '11': 'RO',
    '12': 'AC',
    '13': 'AM',
    '14': 'RR',
    '15': 'PA',
    '16': 'AP',
    '17': 'TO',

    '21': 'MA',
    '22': 'PI',
    '23': 'CE',
    '24': 'RN',
    '25': 'PB',
    '26': 'PE',
    '27': 'AL',
    '28': 'SE',
    '29': 'BA',

    '31': 'MG',
    '32': 'ES',
    '33': 'RJ',
    '35': 'SP',

    '41': 'PR',
    '42': 'SC',
    '43': 'RS',

    '50': 'MS',
    '51': 'MT',
    '52': 'GO',
    '53': 'DF'
}


CHUNKSIZE = 500_000


os.makedirs(
    PASTA_TEMP,
    exist_ok=True
)

os.makedirs(
    PASTA_SAIDA,
    exist_ok=True
)


# ============================================================
# 2. NORMALIZAR TEXTO
# ============================================================

def normalizar_texto(texto):

    texto = str(
        texto
    )

    texto = unicodedata.normalize(
        'NFKD',
        texto
    )

    texto = ''.join(
        caractere
        for caractere in texto
        if not unicodedata.combining(
            caractere
        )
    )

    texto = texto.upper()

    texto = re.sub(
        r'[^A-Z0-9]+',
        ' ',
        texto
    )

    texto = re.sub(
        r'\s+',
        ' ',
        texto
    ).strip()

    return texto


# ============================================================
# 3. IDENTIFICAR LAYOUT
# ============================================================

def identificar_layout(caminho):

    separadores = [
        ';',
        ',',
        '\t',
        '|'
    ]

    encodings = [
        'utf-8',
        'cp1252',
        'latin1'
    ]


    for encoding in encodings:

        for separador in separadores:

            try:

                cabecalho = pd.read_csv(

                    caminho,

                    sep=separador,

                    encoding=encoding,

                    nrows=0
                )


                colunas = list(
                    cabecalho.columns
                )


                if len(colunas) < 5:

                    continue


                cbo_coluna = None

                municipio_coluna = None


                # ------------------------------------------------
                # Identificar CBO
                # ------------------------------------------------

                for coluna in colunas:

                    nome = normalizar_texto(
                        coluna
                    )

                    if (
                        'CBO' in nome
                        and
                        '2002' in nome
                        and
                        'OCUP' in nome
                    ):

                        cbo_coluna = coluna

                        break


                # ------------------------------------------------
                # Identificar Município de Trabalho
                # ------------------------------------------------

                for coluna in colunas:

                    nome = normalizar_texto(
                        coluna
                    )

                    if (
                        'MUN' in nome
                        and
                        'TRAB' in nome
                    ):

                        municipio_coluna = coluna

                        break


                if cbo_coluna is None:

                    continue


                return {

                    'encoding':
                        encoding,

                    'sep':
                        separador,

                    'cbo':
                        cbo_coluna,

                    'municipio':
                        municipio_coluna
                }


            except Exception:

                continue


    raise RuntimeError(
        f'Não foi possível identificar '
        f'o layout de {caminho}'
    )


# ============================================================
# 4. LOCALIZAR .7Z
# ============================================================

def localizar_7z(
    ano,
    grupo
):

    pasta = os.path.join(
        PASTA_BASE,
        grupo,
        str(ano)
    )


    arquivos = glob.glob(
        os.path.join(
            pasta,
            '*.7z'
        )
    )


    if len(arquivos) == 0:

        arquivos = glob.glob(
            os.path.join(
                pasta,
                '*.7Z'
            )
        )


    if len(arquivos) != 1:

        raise RuntimeError(
            f'Esperado exatamente 1 arquivo .7z '
            f'em {pasta}. '
            f'Encontrados: {arquivos}'
        )


    return arquivos[0]


# ============================================================
# 5. CONTADORES
# ============================================================

contagem = Counter()

resumo_processamento = []


# ============================================================
# 6. PROCESSAR ARQUIVOS BRUTOS
# ============================================================

for ano in ANOS:

    for grupo in GRUPOS:

        print('\n' + '#' * 80)

        print(
            f'{ano} - {grupo}'
        )

        print('#' * 80)


        pasta_temp = os.path.join(
            PASTA_TEMP,
            str(ano),
            grupo
        )


        try:

            # ------------------------------------------------
            # Localizar compactado
            # ------------------------------------------------

            arquivo_7z = localizar_7z(
                ano,
                grupo
            )


            print(
                'Arquivo: '
                f'{os.path.basename(arquivo_7z)}'
            )


            # ------------------------------------------------
            # Limpar temporário
            # ------------------------------------------------

            if os.path.exists(
                pasta_temp
            ):

                shutil.rmtree(
                    pasta_temp
                )


            os.makedirs(
                pasta_temp,
                exist_ok=True
            )


            # ------------------------------------------------
            # Extrair
            # ------------------------------------------------

            print(
                'Descompactando...'
            )


            subprocess.run(

                [
                    '7z',
                    'x',
                    arquivo_7z,
                    f'-o{pasta_temp}',
                    '-y'
                ],

                check=True,

                stdout=subprocess.DEVNULL
            )


            # ------------------------------------------------
            # Localizar arquivo extraído
            # ------------------------------------------------

            extraidos = []


            for raiz, _, arquivos in os.walk(
                pasta_temp
            ):

                for nome in arquivos:

                    caminho = os.path.join(
                        raiz,
                        nome
                    )

                    if os.path.isfile(
                        caminho
                    ):

                        extraidos.append(
                            caminho
                        )


            if len(extraidos) != 1:

                raise RuntimeError(
                    f'Esperado 1 arquivo extraído. '
                    f'Encontrados: {extraidos}'
                )


            bruto = extraidos[0]


            layout = identificar_layout(
                bruto
            )


            print(
                f'CBO: {layout["cbo"]}'
            )

            print(
                f'Município: '
                f'{layout["municipio"]}'
            )

            print(
                f'Separador: '
                f'{repr(layout["sep"])}'
            )

            print(
                f'Encoding: '
                f'{layout["encoding"]}'
            )


            # ------------------------------------------------
            # Ler somente colunas necessárias
            # ------------------------------------------------

            colunas = [
                layout['cbo']
            ]


            if (
                layout['municipio']
                is not None
            ):

                colunas.append(
                    layout['municipio']
                )


            leitor = pd.read_csv(

                bruto,

                sep=layout['sep'],

                encoding=layout[
                    'encoding'
                ],

                usecols=colunas,

                dtype=str,

                chunksize=CHUNKSIZE,

                low_memory=False
            )


            total_linhas = 0

            total_5_familias = 0


            for numero_chunk, chunk in enumerate(
                leitor,
                start=1
            ):

                total_linhas += len(
                    chunk
                )


                # --------------------------------------------
                # CBO
                # --------------------------------------------

                cbo = (

                    chunk[
                        layout['cbo']
                    ]

                    .astype(
                        'string'
                    )

                    .str.extract(
                        r'(\d{6})',
                        expand=False
                    )
                )


                familia = (
                    cbo.str[:4]
                )


                mascara = (
                    familia.isin(
                        FAMILIAS.keys()
                    )
                )


                if mascara.sum() == 0:

                    continue


                selecionados = (
                    chunk.loc[
                        mascara
                    ].copy()
                )


                selecionados[
                    'Familia_CBO'
                ] = (

                    familia.loc[
                        mascara
                    ]
                    .values
                )


                selecionados[
                    'CBO_6digitos'
                ] = (

                    cbo.loc[
                        mascara
                    ]
                    .values
                )


                # --------------------------------------------
                # UF pelo código do município
                # --------------------------------------------

                if (
                    layout['municipio']
                    is not None
                ):

                    municipio = (

                        selecionados[
                            layout[
                                'municipio'
                            ]
                        ]

                        .astype(
                            'string'
                        )

                        .str.extract(
                            r'(\d{7})',
                            expand=False
                        )
                    )


                    codigo_uf = (
                        municipio.str[:2]
                    )


                    selecionados[
                        'UF'
                    ] = (

                        codigo_uf.map(
                            MAPA_UF
                        )
                    )


                else:

                    selecionados[
                        'UF'
                    ] = pd.NA


                # --------------------------------------------
                # Contar
                # --------------------------------------------

                grupos_chunk = (

                    selecionados

                    .groupby(
                        [
                            'UF',
                            'Familia_CBO'
                        ],
                        dropna=False
                    )

                    .size()
                )


                for (
                    uf,
                    familia_cbo
                ), qtd in grupos_chunk.items():

                    contagem[
                        (
                            ano,
                            grupo,
                            uf,
                            familia_cbo
                        )
                    ] += int(qtd)


                total_5_familias += len(
                    selecionados
                )


                print(
                    f'Chunk {numero_chunk}: '
                    f'{total_linhas:,} linhas lidas | '
                    f'{total_5_familias:,} '
                    f'nas 5 famílias'
                )


            resumo_processamento.append({

                'Ano':
                    ano,

                'Grupo':
                    grupo,

                'Linhas_lidas':
                    total_linhas,

                'Registros_5_familias':
                    total_5_familias,

                'Status':
                    'OK'
            })


            # ------------------------------------------------
            # Limpar bruto temporário
            # ------------------------------------------------

            shutil.rmtree(
                pasta_temp
            )


        except Exception as erro:

            print(
                f'ERRO: {erro}'
            )


            resumo_processamento.append({

                'Ano':
                    ano,

                'Grupo':
                    grupo,

                'Linhas_lidas':
                    None,

                'Registros_5_familias':
                    None,

                'Status':
                    str(erro)
            })


            if os.path.exists(
                pasta_temp
            ):

                shutil.rmtree(
                    pasta_temp
                )


# ============================================================
# 7. TRANSFORMAR RESULTADOS
# ============================================================

dados = []


for (
    ano,
    grupo,
    uf,
    familia
), qtd in contagem.items():

    dados.append({

        'Ano':
            ano,

        'Grupo':
            grupo,

        'UF':
            uf,

        'Familia_CBO':
            familia,

        'Descricao':
            FAMILIAS.get(
                familia
            ),

        'Professores':
            qtd
    })


df_5_familias = pd.DataFrame(
    dados
)


df_5_familias = (

    df_5_familias

    .sort_values(
        [
            'Ano',
            'UF',
            'Familia_CBO'
        ]
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 8. RESUMO BRASIL POR FAMÍLIA
# ============================================================

df_5_brasil = (

    df_5_familias

    .groupby(
        [
            'Ano',
            'Familia_CBO',
            'Descricao'
        ],
        as_index=False
    )

    ['Professores']

    .sum()
)


# ============================================================
# 9. COMPARAR FILTRO ATUAL x FILTRO AMPLIADO
# ============================================================

df_comparacao = (

    df_5_familias

    .pivot_table(

        index=[
            'Ano',
            'UF'
        ],

        columns=
            'Familia_CBO',

        values=
            'Professores',

        aggfunc=
            'sum',

        fill_value=0
    )

    .reset_index()
)


# Garantir que todas as colunas existam
for familia in [
    '2312',
    '2313',
    '2321',
    '3312',
    '3321'
]:

    if familia not in (
        df_comparacao.columns
    ):

        df_comparacao[
            familia
        ] = 0


df_comparacao[
    'Filtro_atual_3_familias'
] = (

    df_comparacao['2312']
    +
    df_comparacao['2313']
    +
    df_comparacao['2321']
)


df_comparacao[
    'Familias_adicionais'
] = (

    df_comparacao['3312']
    +
    df_comparacao['3321']
)


df_comparacao[
    'Filtro_ampliado_5_familias'
] = (

    df_comparacao[
        'Filtro_atual_3_familias'
    ]

    +
    df_comparacao[
        'Familias_adicionais'
    ]
)


df_comparacao[
    'Percentual_adicional'
] = (

    df_comparacao[
        'Familias_adicionais'
    ]

    /

    df_comparacao[
        'Filtro_ampliado_5_familias'
    ]

    * 100
)


# ============================================================
# 10. RESUMO NACIONAL DA COMPARAÇÃO
# ============================================================

df_comparacao_brasil = (

    df_comparacao

    .groupby(
        'Ano',
        as_index=False
    )

    [
        [
            '2312',
            '2313',
            '2321',
            '3312',
            '3321',
            'Filtro_atual_3_familias',
            'Familias_adicionais',
            'Filtro_ampliado_5_familias'
        ]
    ]

    .sum()
)


df_comparacao_brasil[
    'Percentual_adicional'
] = (

    df_comparacao_brasil[
        'Familias_adicionais'
    ]

    /

    df_comparacao_brasil[
        'Filtro_ampliado_5_familias'
    ]

    * 100
)


# ============================================================
# 11. SALVAR
# ============================================================

df_5_familias.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '08_cinco_familias_por_ano_uf.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_5_brasil.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '09_cinco_familias_brasil.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_comparacao.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '10_comparacao_3_vs_5_familias_uf.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_comparacao_brasil.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '11_comparacao_3_vs_5_familias_brasil.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


pd.DataFrame(
    resumo_processamento
).to_csv(

    os.path.join(
        PASTA_SAIDA,
        '12_status_auditoria_brutos.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 12. RESULTADOS PRINCIPAIS
# ============================================================

print('\n' + '=' * 80)
print('COMPARAÇÃO NACIONAL')
print('=' * 80)

display(
    df_comparacao_brasil
)


print('\n' + '=' * 80)
print('COMPARAÇÃO POR UF')
print('=' * 80)

display(
    df_comparacao
)


print(
    '\nResultados salvos em:\n'
    f'{PASTA_SAIDA}'
)

In [ ]:
# ============================================================
# AUDITORIA DEFINITIVA DOS AFASTAMENTOS
# RAIS - PROFESSORES - 5 FAMÍLIAS CBO
#
# Anos:
#   2020, 2021, 2022, 2023, 2024, 2025
#
# OBJETIVO:
#
#   Construir e auditar a variável-alvo:
#
#       Y_doenca = 1
#
#   quando pelo menos uma das três causas de afastamento
#   possuir código:
#
#       30 = doença relacionada ao trabalho
#       40 = doença não relacionada ao trabalho
#
#   Caso contrário:
#
#       Y_doenca = 0
#
#
# IMPORTANTE:
#
#   - 99 e 999 NÃO são tratados como afastamento.
#
#   - "Qtd Dias Afastamento" NÃO define Y_doenca.
#
#   - A quantidade de dias será utilizada apenas para
#     auditoria de consistência dos dados.
#
#   - Este código NÃO altera os Parquets originais.
#
# ============================================================


# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

from google.colab import drive

import os
import re
import glob
import unicodedata

from collections import Counter

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

drive.mount(
    '/content/drive',
    force_remount=False
)


# ============================================================
# 3. PASTAS
# ============================================================

PASTA_PARQUET = (
    '/content/drive/MyDrive/TCC_2/dados/'
    'RAIS_PROFESSORES_5CBO'
)


PASTA_SAIDA = (
    '/content/drive/MyDrive/TCC_2/resultados/'
    'AUDITORIA_DOENCA_DEFINITIVA'
)


os.makedirs(
    PASTA_SAIDA,
    exist_ok=True
)


# ============================================================
# 4. CONFIGURAÇÕES
# ============================================================

ANOS = [
    2020,
    2021,
    2022,
    2023,
    2024,
    2025
]


# ------------------------------------------------------------
# Códigos que formarão a variável-alvo
# ------------------------------------------------------------

CODIGOS_DOENCA = {
    30,
    40
}


# ------------------------------------------------------------
# Marcadores observados nos layouts da RAIS
#
# Eles NÃO serão considerados causas reais.
# ------------------------------------------------------------

CODIGOS_SENTINELA_GERAIS = {
    99,
    999
}


# ------------------------------------------------------------
# Sentinela esperada por layout/ano
#
# Isso é usado apenas para auditoria adicional.
# ------------------------------------------------------------

SENTINELA_ESPERADA = {

    2020: 99,
    2021: 99,
    2022: 99,

    2023: 999,
    2024: 999,
    2025: 999
}


# ------------------------------------------------------------
# Tamanho dos lotes
# ------------------------------------------------------------

BATCH_SIZE = 250_000


# ============================================================
# 5. NORMALIZAR TEXTO
# ============================================================

def normalizar_texto(texto):

    texto = str(
        texto
    )

    texto = unicodedata.normalize(
        'NFKD',
        texto
    )

    texto = ''.join(

        caractere

        for caractere in texto

        if not unicodedata.combining(
            caractere
        )
    )

    texto = texto.upper()

    texto = re.sub(
        r'[^A-Z0-9]+',
        ' ',
        texto
    )

    texto = re.sub(
        r'\s+',
        ' ',
        texto
    ).strip()

    return texto


# ============================================================
# 6. LIMPAR CÓDIGOS DAS CAUSAS
# ============================================================

def limpar_codigo(serie):

    texto = (

        serie
        .astype('string')
        .str.strip()
        .str.replace(
            ',',
            '.',
            regex=False
        )
    )


    numero = pd.to_numeric(
        texto,
        errors='coerce'
    )


    # --------------------------------------------------------
    # Aceitar somente códigos inteiros
    # --------------------------------------------------------

    inteiro = (

        numero.isna()

        |

        np.isclose(
            numero,
            numero.round(),
            equal_nan=True
        )
    )


    numero = numero.where(
        inteiro
    )


    return (

        numero
        .round()
        .astype('Int64')
    )


# ============================================================
# 7. LIMPAR QUANTIDADE DE DIAS
# ============================================================

def limpar_dias(serie):

    texto = (

        serie
        .astype('string')
        .str.strip()
        .str.replace(
            ',',
            '.',
            regex=False
        )
    )


    return pd.to_numeric(
        texto,
        errors='coerce'
    )


# ============================================================
# 8. IDENTIFICAR ANO E GRUPO
# ============================================================

def identificar_arquivo(caminho):

    nome = os.path.basename(
        caminho
    )


    padrao = (
        r'RAIS_PROFESSORES_5CBO_'
        r'(\d{4})_(.+)\.parquet$'
    )


    resultado = re.search(
        padrao,
        nome,
        flags=re.IGNORECASE
    )


    if resultado is None:

        raise RuntimeError(
            f'Nome de arquivo não reconhecido: {nome}'
        )


    ano = int(
        resultado.group(1)
    )


    grupo = (
        resultado
        .group(2)
        .upper()
    )


    return ano, grupo


# ============================================================
# 9. IDENTIFICAR COLUNAS
# ============================================================

def identificar_colunas(colunas):

    mapa = {

        coluna:
            normalizar_texto(
                coluna
            )

        for coluna in colunas
    }


    # ========================================================
    # CAUSAS 1, 2 e 3
    # ========================================================

    causas = {}


    for numero in [
        1,
        2,
        3
    ]:

        candidatos = []


        for coluna, nome in mapa.items():

            if (

                'CAUSA'
                in nome

                and

                'AFASTAMENTO'
                in nome

                and

                re.search(
                    rf'\b{numero}\b',
                    nome
                )
            ):

                candidatos.append(
                    coluna
                )


        # ----------------------------------------------------
        # Novo layout pode ter "Código".
        # Preferir essa coluna.
        # ----------------------------------------------------

        if len(
            candidatos
        ) > 1:

            candidatos_codigo = [

                coluna

                for coluna
                in candidatos

                if (
                    'CODIGO'
                    in mapa[coluna]
                )
            ]


            if len(
                candidatos_codigo
            ) == 1:

                candidatos = (
                    candidatos_codigo
                )


        if len(
            candidatos
        ) != 1:

            raise RuntimeError(

                f'Não consegui identificar '
                f'Causa Afastamento {numero}.\n'

                f'Candidatos: {candidatos}'
            )


        causas[
            numero
        ] = candidatos[0]


    # ========================================================
    # QUANTIDADE DE DIAS
    # ========================================================

    candidatos_dias = []


    for coluna, nome in mapa.items():

        if (

            'QTD'
            in nome

            and

            'DIAS'
            in nome

            and

            'AFASTAMENTO'
            in nome
        ):

            candidatos_dias.append(
                coluna
            )


    if len(
        candidatos_dias
    ) != 1:

        raise RuntimeError(

            'Não consegui identificar '
            'Qtd Dias Afastamento.\n'

            f'Candidatos: {candidatos_dias}'
        )


    coluna_dias = (
        candidatos_dias[0]
    )


    # ========================================================
    # NATUREZA JURÍDICA
    # ========================================================

    candidatos_natureza = []


    for coluna, nome in mapa.items():

        if (

            'NATUREZA'
            in nome

            and

            'JURIDICA'
            in nome
        ):

            candidatos_natureza.append(
                coluna
            )


    coluna_natureza = None


    if len(
        candidatos_natureza
    ) > 0:

        # ----------------------------------------------------
        # Preferir coluna de código
        # ----------------------------------------------------

        candidatos_codigo = [

            coluna

            for coluna
            in candidatos_natureza

            if (
                'CODIGO'
                in mapa[coluna]
            )
        ]


        if len(
            candidatos_codigo
        ) == 1:

            coluna_natureza = (
                candidatos_codigo[0]
            )


        elif len(
            candidatos_natureza
        ) == 1:

            coluna_natureza = (
                candidatos_natureza[0]
            )


        else:

            coluna_natureza = (
                candidatos_natureza[0]
            )


    return {

        'causa1':
            causas[1],

        'causa2':
            causas[2],

        'causa3':
            causas[3],

        'dias':
            coluna_dias,

        'natureza':
            coluna_natureza
    }


# ============================================================
# 10. IDENTIFICAR CAUSA REAL
# ============================================================

def causa_valida(serie):

    # --------------------------------------------------------
    # Uma causa é considerada "válida" se:
    #
    # - não for nula
    # - não for zero
    # - não for 99
    # - não for 999
    #
    # Isso NÃO significa doença.
    #
    # Apenas significa que existe algum motivo de afastamento.
    # --------------------------------------------------------

    return (

        serie.notna()

        &

        (~serie.isin(
            CODIGOS_SENTINELA_GERAIS
        ))

        &

        serie.ne(0)
    )


# ============================================================
# 11. LOCALIZAR PARQUETS
# ============================================================

arquivos_parquet = sorted(

    glob.glob(

        os.path.join(
            PASTA_PARQUET,
            '**',
            '*.parquet'
        ),

        recursive=True
    )
)


print(
    f'Parquets encontrados: '
    f'{len(arquivos_parquet)}'
)


if len(
    arquivos_parquet
) != 36:

    print(
        '\nATENÇÃO: eram esperados 36 Parquets '
        '(6 anos x 6 grupos).'
    )


# ============================================================
# 12. COLUNAS NUMÉRICAS DE RESUMO
# ============================================================

COLUNAS_SOMA = [

    'N',

    'Doenca',

    'Doenca_30',

    'Doenca_40',

    'Doenca_30_e_40',

    'Qualquer_causa_valida',

    'Sem_causa_valida',

    'Todas_causas_nulas',

    'Sentinela_esperada_alguma',

    'Sentinela_inesperada_alguma',

    'Dias_nulo',

    'Dias_zero',

    'Dias_positivo',

    'Dias_negativo',

    'Doenca_Dias_positivo',

    'Doenca_Dias_zero',

    'Doenca_Dias_nulo',

    'Nao_doenca_Dias_positivo',

    'Nao_doenca_Dias_zero',

    'Nao_doenca_Dias_nulo',

    'Causa_valida_Dias_positivo',

    'Causa_valida_Dias_zero',

    'Causa_valida_Dias_nulo',

    'Soma_dias_positivos'
]


# ============================================================
# 13. CRIAR RESUMO DE UM BATCH
# ============================================================

def resumir_batch(
    base,
    dimensoes
):

    agregacoes = {

        coluna:
            (
                coluna,
                'sum'
            )

        for coluna
        in COLUNAS_SOMA
    }


    agregacoes[
        'Max_dias'
    ] = (
        'Dias_valor',
        'max'
    )


    resumo = (

        base

        .groupby(
            dimensoes,
            dropna=False,
            observed=True
        )

        .agg(
            **agregacoes
        )

        .reset_index()
    )


    return resumo


# ============================================================
# 14. CONSOLIDAR RESUMOS
# ============================================================

def consolidar_resumos(
    lista,
    dimensoes
):

    if len(
        lista
    ) == 0:

        return pd.DataFrame()


    base = pd.concat(
        lista,
        ignore_index=True
    )


    agregacoes = {

        coluna:
            (
                coluna,
                'sum'
            )

        for coluna
        in COLUNAS_SOMA
    }


    agregacoes[
        'Max_dias'
    ] = (
        'Max_dias',
        'max'
    )


    resultado = (

        base

        .groupby(
            dimensoes,
            dropna=False,
            observed=True
        )

        .agg(
            **agregacoes
        )

        .reset_index()
    )


    return adicionar_percentuais(
        resultado
    )


# ============================================================
# 15. ADICIONAR PERCENTUAIS
# ============================================================

def adicionar_percentuais(df):

    df = df.copy()


    def percentual(
        numerador,
        denominador
    ):

        return np.where(

            denominador > 0,

            numerador
            /
            denominador
            *
            100,

            np.nan
        )


    # --------------------------------------------------------
    # PREVALÊNCIA DO DESFECHO
    # --------------------------------------------------------

    df[
        'Pct_doenca'
    ] = percentual(

        df[
            'Doenca'
        ],

        df[
            'N'
        ]
    )


    df[
        'Pct_doenca_30'
    ] = percentual(

        df[
            'Doenca_30'
        ],

        df[
            'N'
        ]
    )


    df[
        'Pct_doenca_40'
    ] = percentual(

        df[
            'Doenca_40'
        ],

        df[
            'N'
        ]
    )


    # --------------------------------------------------------
    # CAUSA VÁLIDA
    # --------------------------------------------------------

    df[
        'Pct_qualquer_causa_valida'
    ] = percentual(

        df[
            'Qualquer_causa_valida'
        ],

        df[
            'N'
        ]
    )


    # --------------------------------------------------------
    # DIAS
    # --------------------------------------------------------

    df[
        'Pct_dias_positivo'
    ] = percentual(

        df[
            'Dias_positivo'
        ],

        df[
            'N'
        ]
    )


    df[
        'Pct_dias_zero'
    ] = percentual(

        df[
            'Dias_zero'
        ],

        df[
            'N'
        ]
    )


    df[
        'Pct_dias_nulo'
    ] = percentual(

        df[
            'Dias_nulo'
        ],

        df[
            'N'
        ]
    )


    # --------------------------------------------------------
    # ENTRE OS CASOS DE DOENÇA
    # --------------------------------------------------------

    df[
        'Pct_doenca_com_dias_positivo'
    ] = percentual(

        df[
            'Doenca_Dias_positivo'
        ],

        df[
            'Doenca'
        ]
    )


    df[
        'Pct_doenca_com_dias_zero'
    ] = percentual(

        df[
            'Doenca_Dias_zero'
        ],

        df[
            'Doenca'
        ]
    )


    df[
        'Pct_doenca_com_dias_nulo'
    ] = percentual(

        df[
            'Doenca_Dias_nulo'
        ],

        df[
            'Doenca'
        ]
    )


    # --------------------------------------------------------
    # ENTRE QUEM TEM DIAS > 0
    # --------------------------------------------------------

    df[
        'Pct_dias_pos_sem_doenca'
    ] = percentual(

        df[
            'Nao_doenca_Dias_positivo'
        ],

        df[
            'Dias_positivo'
        ]
    )


    # --------------------------------------------------------
    # ENTRE QUEM TEM ALGUMA CAUSA VÁLIDA
    # --------------------------------------------------------

    df[
        'Pct_causa_valida_com_dias_zero'
    ] = percentual(

        df[
            'Causa_valida_Dias_zero'
        ],

        df[
            'Qualquer_causa_valida'
        ]
    )


    df[
        'Pct_doenca_entre_causas_validas'
    ] = percentual(

        df[
            'Doenca'
        ],

        df[
            'Qualquer_causa_valida'
        ]
    )


    # --------------------------------------------------------
    # SENTINELAS
    # --------------------------------------------------------

    df[
        'Pct_sentinela_esperada'
    ] = percentual(

        df[
            'Sentinela_esperada_alguma'
        ],

        df[
            'N'
        ]
    )


    df[
        'Pct_sentinela_inesperada'
    ] = percentual(

        df[
            'Sentinela_inesperada_alguma'
        ],

        df[
            'N'
        ]
    )


    # --------------------------------------------------------
    # MÉDIA DE DIAS POSITIVOS
    # --------------------------------------------------------

    df[
        'Media_dias_positivos'
    ] = np.where(

        df[
            'Dias_positivo'
        ] > 0,

        df[
            'Soma_dias_positivos'
        ]

        /

        df[
            'Dias_positivo'
        ],

        np.nan
    )


    return df


# ============================================================
# 16. LISTAS DE RESUMOS
# ============================================================

resumos_ano = []

resumos_ano_uf = []

resumos_ano_familia = []

resumos_ano_natureza = []


# ============================================================
# 17. CONTAGEM DOS CÓDIGOS BRUTOS
# ============================================================

contagem_codigos = Counter()


# ============================================================
# 18. STATUS DOS ARQUIVOS
# ============================================================

status_arquivos = []


# ============================================================
# 19. PROCESSAMENTO DOS 36 PARQUETS
# ============================================================

for indice, arquivo in enumerate(

    arquivos_parquet,

    start=1
):

    print(
        '\n'
        +
        '=' * 90
    )


    try:

        # ----------------------------------------------------
        # Ano e grupo
        # ----------------------------------------------------

        ano, grupo = (
            identificar_arquivo(
                arquivo
            )
        )


        print(

            f'[{indice}/'
            f'{len(arquivos_parquet)}] '

            f'{ano} - {grupo}'
        )


        # ----------------------------------------------------
        # Parquet
        # ----------------------------------------------------

        parquet = pq.ParquetFile(
            arquivo
        )


        colunas = (
            parquet.schema.names
        )


        layout = (
            identificar_colunas(
                colunas
            )
        )


        print(
            f'Causa 1: '
            f'{layout["causa1"]}'
        )

        print(
            f'Causa 2: '
            f'{layout["causa2"]}'
        )

        print(
            f'Causa 3: '
            f'{layout["causa3"]}'
        )

        print(
            f'Dias: '
            f'{layout["dias"]}'
        )

        print(
            f'Natureza: '
            f'{layout["natureza"]}'
        )


        # ----------------------------------------------------
        # Verificações
        # ----------------------------------------------------

        for coluna in [

            'UF',
            'Familia_CBO'

        ]:

            if coluna not in colunas:

                raise RuntimeError(

                    f'Coluna obrigatória '
                    f'ausente: {coluna}'
                )


        # ----------------------------------------------------
        # Ler somente o necessário
        # ----------------------------------------------------

        colunas_leitura = [

            'UF',

            'Familia_CBO',

            layout[
                'causa1'
            ],

            layout[
                'causa2'
            ],

            layout[
                'causa3'
            ],

            layout[
                'dias'
            ]
        ]


        if (
            layout[
                'natureza'
            ]
            is not None
        ):

            colunas_leitura.append(

                layout[
                    'natureza'
                ]
            )


        linhas_arquivo = 0


        # ====================================================
        # BATCHES
        # ====================================================

        for numero_batch, batch in enumerate(

            parquet.iter_batches(

                batch_size=
                    BATCH_SIZE,

                columns=
                    colunas_leitura
            ),

            start=1
        ):

            df = batch.to_pandas()


            linhas_arquivo += len(
                df
            )


            # =================================================
            # PADRONIZAÇÕES
            # =================================================

            uf = (

                df[
                    'UF'
                ]

                .astype(
                    'string'
                )

                .str.strip()

                .str.upper()
            )


            familia = (

                df[
                    'Familia_CBO'
                ]

                .astype(
                    'string'
                )

                .str.extract(
                    r'(\d{4})',
                    expand=False
                )
            )


            causa1 = limpar_codigo(

                df[
                    layout[
                        'causa1'
                    ]
                ]
            )


            causa2 = limpar_codigo(

                df[
                    layout[
                        'causa2'
                    ]
                ]
            )


            causa3 = limpar_codigo(

                df[
                    layout[
                        'causa3'
                    ]
                ]
            )


            dias = limpar_dias(

                df[
                    layout[
                        'dias'
                    ]
                ]
            )


            # =================================================
            # CAUSAS VÁLIDAS
            # =================================================

            valida1 = causa_valida(
                causa1
            )

            valida2 = causa_valida(
                causa2
            )

            valida3 = causa_valida(
                causa3
            )


            qualquer_causa_valida = (

                valida1
                |
                valida2
                |
                valida3
            )


            # =================================================
            # VARIÁVEL-ALVO
            #
            # Y_doenca = 1
            # se causa 1, 2 ou 3 for 30 ou 40.
            #
            # NÃO utiliza quantidade de dias.
            # =================================================

            doenca_30 = (

                causa1.eq(30).fillna(False)

                |

                causa2.eq(30).fillna(False)

                |

                causa3.eq(30).fillna(False)
            )


            doenca_40 = (

                causa1.eq(40).fillna(False)

                |

                causa2.eq(40).fillna(False)

                |

                causa3.eq(40).fillna(False)
            )


            y_doenca = (

                doenca_30
                |
                doenca_40
            )


            doenca_30_e_40 = (

                doenca_30
                &
                doenca_40
            )


            # =================================================
            # SENTINELAS
            # =================================================

            sentinela_ano = (
                SENTINELA_ESPERADA[
                    ano
                ]
            )


            sentinela_esperada = (

                causa1.eq(
                    sentinela_ano
                ).fillna(False)

                |

                causa2.eq(
                    sentinela_ano
                ).fillna(False)

                |

                causa3.eq(
                    sentinela_ano
                ).fillna(False)
            )


            outras_sentinelas = (

                CODIGOS_SENTINELA_GERAIS
                -
                {
                    sentinela_ano
                }
            )


            sentinela_inesperada = (

                causa1.isin(
                    outras_sentinelas
                )

                |

                causa2.isin(
                    outras_sentinelas
                )

                |

                causa3.isin(
                    outras_sentinelas
                )
            )


            # =================================================
            # CAUSAS NULAS
            # =================================================

            todas_causas_nulas = (

                causa1.isna()

                &

                causa2.isna()

                &

                causa3.isna()
            )


            # =================================================
            # DIAS
            # =================================================

            dias_nulo = (
                dias.isna()
            )


            dias_zero = (

                dias.eq(0)
                .fillna(False)
            )


            dias_positivo = (

                dias.gt(0)
                .fillna(False)
            )


            dias_negativo = (

                dias.lt(0)
                .fillna(False)
            )


            nao_doenca = (
                ~y_doenca
            )


            # =================================================
            # BASE DE MÉTRICAS
            # =================================================

            base = pd.DataFrame({

                'Ano':
                    ano,

                'UF':
                    uf,

                'Familia_CBO':
                    familia,

                'N':
                    1,

                'Doenca':
                    y_doenca.astype(
                        'int8'
                    ),

                'Doenca_30':
                    doenca_30.astype(
                        'int8'
                    ),

                'Doenca_40':
                    doenca_40.astype(
                        'int8'
                    ),

                'Doenca_30_e_40':
                    doenca_30_e_40.astype(
                        'int8'
                    ),

                'Qualquer_causa_valida':
                    qualquer_causa_valida.astype(
                        'int8'
                    ),

                'Sem_causa_valida':
                    (
                        ~qualquer_causa_valida
                    ).astype(
                        'int8'
                    ),

                'Todas_causas_nulas':
                    todas_causas_nulas.astype(
                        'int8'
                    ),

                'Sentinela_esperada_alguma':
                    sentinela_esperada.astype(
                        'int8'
                    ),

                'Sentinela_inesperada_alguma':
                    sentinela_inesperada.astype(
                        'int8'
                    ),

                'Dias_nulo':
                    dias_nulo.astype(
                        'int8'
                    ),

                'Dias_zero':
                    dias_zero.astype(
                        'int8'
                    ),

                'Dias_positivo':
                    dias_positivo.astype(
                        'int8'
                    ),

                'Dias_negativo':
                    dias_negativo.astype(
                        'int8'
                    ),

                'Doenca_Dias_positivo':
                    (
                        y_doenca
                        &
                        dias_positivo
                    ).astype(
                        'int8'
                    ),

                'Doenca_Dias_zero':
                    (
                        y_doenca
                        &
                        dias_zero
                    ).astype(
                        'int8'
                    ),

                'Doenca_Dias_nulo':
                    (
                        y_doenca
                        &
                        dias_nulo
                    ).astype(
                        'int8'
                    ),

                'Nao_doenca_Dias_positivo':
                    (
                        nao_doenca
                        &
                        dias_positivo
                    ).astype(
                        'int8'
                    ),

                'Nao_doenca_Dias_zero':
                    (
                        nao_doenca
                        &
                        dias_zero
                    ).astype(
                        'int8'
                    ),

                'Nao_doenca_Dias_nulo':
                    (
                        nao_doenca
                        &
                        dias_nulo
                    ).astype(
                        'int8'
                    ),

                'Causa_valida_Dias_positivo':
                    (
                        qualquer_causa_valida
                        &
                        dias_positivo
                    ).astype(
                        'int8'
                    ),

                'Causa_valida_Dias_zero':
                    (
                        qualquer_causa_valida
                        &
                        dias_zero
                    ).astype(
                        'int8'
                    ),

                'Causa_valida_Dias_nulo':
                    (
                        qualquer_causa_valida
                        &
                        dias_nulo
                    ).astype(
                        'int8'
                    ),

                'Soma_dias_positivos':
                    (
                        dias
                        .where(
                            dias_positivo,
                            0
                        )
                        .fillna(0)
                    ),

                'Dias_valor':
                    dias
            })


            # =================================================
            # NATUREZA JURÍDICA
            # =================================================

            if (
                layout[
                    'natureza'
                ]
                is not None
            ):

                natureza = (

                    df[
                        layout[
                            'natureza'
                        ]
                    ]

                    .astype(
                        'string'
                    )

                    .str.strip()
                )


                base[
                    'Natureza_Juridica'
                ] = natureza


            # =================================================
            # RESUMO NACIONAL
            # =================================================

            resumos_ano.append(

                resumir_batch(

                    base,

                    [
                        'Ano'
                    ]
                )
            )


            # =================================================
            # RESUMO POR UF
            # =================================================

            resumos_ano_uf.append(

                resumir_batch(

                    base,

                    [
                        'Ano',
                        'UF'
                    ]
                )
            )


            # =================================================
            # RESUMO POR FAMÍLIA
            # =================================================

            resumos_ano_familia.append(

                resumir_batch(

                    base,

                    [
                        'Ano',
                        'Familia_CBO'
                    ]
                )
            )


            # =================================================
            # RESUMO POR NATUREZA JURÍDICA
            # =================================================

            if (
                'Natureza_Juridica'
                in base.columns
            ):

                resumos_ano_natureza.append(

                    resumir_batch(

                        base,

                        [
                            'Ano',
                            'Natureza_Juridica'
                        ]
                    )
                )


            # =================================================
            # CONTAGEM DOS CÓDIGOS BRUTOS
            # =================================================

            for posicao, serie in [

                (
                    1,
                    causa1
                ),

                (
                    2,
                    causa2
                ),

                (
                    3,
                    causa3
                )
            ]:

                contagens = (

                    serie

                    .astype(
                        'string'
                    )

                    .fillna(
                        '<NA>'
                    )

                    .value_counts(
                        dropna=False
                    )
                )


                for codigo, quantidade in (
                    contagens.items()
                ):

                    contagem_codigos[
                        (
                            ano,
                            posicao,
                            str(
                                codigo
                            )
                        )
                    ] += int(
                        quantidade
                    )


            print(

                f'Batch {numero_batch}: '

                f'{linhas_arquivo:,} '
                f'linhas acumuladas'
            )


        # ====================================================
        # STATUS DO ARQUIVO
        # ====================================================

        status_arquivos.append({

            'Ano':
                ano,

            'Grupo':
                grupo,

            'Arquivo':
                os.path.basename(
                    arquivo
                ),

            'Linhas':
                linhas_arquivo,

            'Causa_1':
                layout[
                    'causa1'
                ],

            'Causa_2':
                layout[
                    'causa2'
                ],

            'Causa_3':
                layout[
                    'causa3'
                ],

            'Qtd_Dias':
                layout[
                    'dias'
                ],

            'Natureza_Juridica':
                layout[
                    'natureza'
                ],

            'Sentinela_esperada':
                SENTINELA_ESPERADA[
                    ano
                ],

            'Status':
                'OK'
        })


    except Exception as erro:

        print(
            f'ERRO: {erro}'
        )


        status_arquivos.append({

            'Ano':
                None,

            'Grupo':
                None,

            'Arquivo':
                os.path.basename(
                    arquivo
                ),

            'Linhas':
                None,

            'Causa_1':
                None,

            'Causa_2':
                None,

            'Causa_3':
                None,

            'Qtd_Dias':
                None,

            'Natureza_Juridica':
                None,

            'Sentinela_esperada':
                None,

            'Status':
                str(
                    erro
                )
        })


# ============================================================
# 20. CONSOLIDAR RESULTADOS
# ============================================================

df_status = pd.DataFrame(
    status_arquivos
)


df_ano = consolidar_resumos(

    resumos_ano,

    [
        'Ano'
    ]
)


df_ano_uf = consolidar_resumos(

    resumos_ano_uf,

    [
        'Ano',
        'UF'
    ]
)


df_ano_familia = consolidar_resumos(

    resumos_ano_familia,

    [
        'Ano',
        'Familia_CBO'
    ]
)


df_ano_natureza = consolidar_resumos(

    resumos_ano_natureza,

    [
        'Ano',
        'Natureza_Juridica'
    ]
)


# ============================================================
# 21. CÓDIGOS DE CAUSA
# ============================================================

registros_codigos = []


for (
    ano,
    posicao,
    codigo
), quantidade in (
    contagem_codigos.items()
):


    # --------------------------------------------------------
    # Classificação auxiliar
    # --------------------------------------------------------

    if codigo == '<NA>':

        tipo_codigo = (
            'AUSENTE'
        )

        eh_doenca = False

        eh_sentinela = False


    else:

        try:

            codigo_num = int(
                codigo
            )

        except Exception:

            codigo_num = None


        if (
            codigo_num
            in CODIGOS_DOENCA
        ):

            tipo_codigo = (
                'DOENCA'
            )

            eh_doenca = True

            eh_sentinela = False


        elif (
            codigo_num
            in CODIGOS_SENTINELA_GERAIS
        ):

            tipo_codigo = (
                'SENTINELA'
            )

            eh_doenca = False

            eh_sentinela = True


        elif (
            codigo_num == 0
        ):

            tipo_codigo = (
                'ZERO'
            )

            eh_doenca = False

            eh_sentinela = False


        else:

            tipo_codigo = (
                'OUTRA_CAUSA'
            )

            eh_doenca = False

            eh_sentinela = False


    registros_codigos.append({

        'Ano':
            ano,

        'Posicao_Causa':
            posicao,

        'Codigo':
            codigo,

        'Quantidade':
            quantidade,

        'Tipo_Codigo':
            tipo_codigo,

        'Eh_Doenca_30_40':
            eh_doenca,

        'Eh_Sentinela_99_999':
            eh_sentinela
    })


df_codigos = pd.DataFrame(
    registros_codigos
)


# ------------------------------------------------------------
# Consolidado das três posições
# ------------------------------------------------------------

df_codigos_consolidado = (

    df_codigos

    .groupby(

        [
            'Ano',
            'Codigo',
            'Tipo_Codigo',
            'Eh_Doenca_30_40',
            'Eh_Sentinela_99_999'
        ],

        as_index=False
    )

    [
        'Quantidade'
    ]

    .sum()
)


# ============================================================
# 22. ORDENAR RESULTADOS
# ============================================================

df_ano = (

    df_ano

    .sort_values(
        'Ano'
    )

    .reset_index(
        drop=True
    )
)


df_ano_uf = (

    df_ano_uf

    .sort_values(
        [
            'UF',
            'Ano'
        ]
    )

    .reset_index(
        drop=True
    )
)


df_ano_familia = (

    df_ano_familia

    .sort_values(
        [
            'Familia_CBO',
            'Ano'
        ]
    )

    .reset_index(
        drop=True
    )
)


df_ano_natureza = (

    df_ano_natureza

    .sort_values(
        [
            'Natureza_Juridica',
            'Ano'
        ]
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 23. VARIAÇÃO ANUAL DO DESFECHO
# ============================================================

df_ano[
    'Variacao_pp_doenca'
] = (

    df_ano[
        'Pct_doenca'
    ]

    .diff()
)


df_ano[
    'Variacao_pct_num_casos'
] = (

    df_ano[
        'Doenca'
    ]

    .pct_change()

    * 100
)


# ============================================================
# 24. COMPARAÇÃO CRÍTICA 2021 x 2022 x 2023
# ============================================================

colunas_comparacao = [

    'Ano',

    'N',

    'Doenca',

    'Pct_doenca',

    'Doenca_30',

    'Pct_doenca_30',

    'Doenca_40',

    'Pct_doenca_40',

    'Qualquer_causa_valida',

    'Pct_qualquer_causa_valida',

    'Dias_positivo',

    'Pct_dias_positivo',

    'Dias_zero',

    'Pct_dias_zero',

    'Doenca_Dias_positivo',

    'Doenca_Dias_zero',

    'Pct_doenca_com_dias_positivo',

    'Pct_doenca_com_dias_zero',

    'Pct_dias_pos_sem_doenca',

    'Media_dias_positivos',

    'Max_dias'
]


df_comparacao_2022 = (

    df_ano

    .loc[

        df_ano[
            'Ano'
        ]

        .isin(
            [
                2021,
                2022,
                2023
            ]
        ),

        colunas_comparacao
    ]

    .reset_index(
        drop=True
    )
)


# ============================================================
# 25. 2022 POR UF
# ============================================================

df_2022_uf = (

    df_ano_uf

    .loc[
        df_ano_uf[
            'Ano'
        ]
        ==
        2022
    ]

    .sort_values(
        'Pct_doenca_com_dias_zero',
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 26. 2022 POR NATUREZA JURÍDICA
# ============================================================

df_2022_natureza = (

    df_ano_natureza

    .loc[
        (
            df_ano_natureza[
                'Ano'
            ]
            ==
            2022
        )

        &

        (
            df_ano_natureza[
                'N'
            ]
            >=
            1000
        )
    ]

    .sort_values(
        'Pct_doenca_com_dias_zero',
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 27. 2022 POR FAMÍLIA CBO
# ============================================================

df_2022_familia = (

    df_ano_familia

    .loc[
        df_ano_familia[
            'Ano'
        ]
        ==
        2022
    ]

    .sort_values(
        'Familia_CBO'
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 28. SALVAR RESULTADOS
# ============================================================

df_status.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '01_status_layout.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_ano.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '02_doenca_por_ano.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_ano_uf.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '03_doenca_ano_uf.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_ano_familia.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '04_doenca_ano_familia_cbo.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_ano_natureza.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '05_doenca_ano_natureza_juridica.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_codigos.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '06_codigos_causa_por_ano_posicao.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_codigos_consolidado.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '07_codigos_causa_consolidado.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_comparacao_2022.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '08_comparacao_2021_2022_2023.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_2022_uf.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '09_2022_por_uf.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_2022_natureza.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '10_2022_por_natureza_juridica.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


df_2022_familia.to_csv(

    os.path.join(
        PASTA_SAIDA,
        '11_2022_por_familia_cbo.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 29. VALIDAÇÃO DOS ARQUIVOS
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    'STATUS DOS ARQUIVOS'
)

print(
    '=' * 90
)


display(
    df_status
)


quantidade_ok = (

    df_status[
        'Status'
    ]
    .eq(
        'OK'
    )
    .sum()
)


print(
    f'\nArquivos OK: '
    f'{quantidade_ok} de 36'
)


# ============================================================
# 30. RESULTADO PRINCIPAL
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    'VARIÁVEL-ALVO: AFASTAMENTO POR DOENÇA'
)

print(
    '=' * 90
)


display(

    df_ano[
        [
            'Ano',

            'N',

            'Doenca',

            'Pct_doenca',

            'Doenca_30',

            'Pct_doenca_30',

            'Doenca_40',

            'Pct_doenca_40',

            'Variacao_pp_doenca',

            'Variacao_pct_num_casos'
        ]
    ]
)


# ============================================================
# 31. COMPARAÇÃO 2021 x 2022 x 2023
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    'COMPARAÇÃO CRÍTICA: 2021 x 2022 x 2023'
)

print(
    '=' * 90
)


display(
    df_comparacao_2022
)


# ============================================================
# 32. 2022 POR UF
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    '2022 - AUDITORIA POR UF'
)

print(
    '=' * 90
)


display(

    df_2022_uf[
        [
            'Ano',
            'UF',

            'N',

            'Doenca',

            'Pct_doenca',

            'Pct_dias_positivo',

            'Pct_doenca_com_dias_positivo',

            'Pct_doenca_com_dias_zero'
        ]
    ]
)


# ============================================================
# 33. 2022 POR FAMÍLIA CBO
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    '2022 - AUDITORIA POR FAMÍLIA CBO'
)

print(
    '=' * 90
)


display(

    df_2022_familia[
        [
            'Familia_CBO',

            'N',

            'Doenca',

            'Pct_doenca',

            'Pct_dias_positivo',

            'Pct_doenca_com_dias_zero'
        ]
    ]
)


# ============================================================
# 34. 2022 POR NATUREZA JURÍDICA
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    '2022 - NATUREZAS JURÍDICAS COM N >= 1.000'
)

print(
    '=' * 90
)


display(

    df_2022_natureza[
        [
            'Natureza_Juridica',

            'N',

            'Doenca',

            'Pct_doenca',

            'Dias_positivo',

            'Pct_dias_positivo',

            'Doenca_Dias_zero',

            'Pct_doenca_com_dias_zero'
        ]
    ]
)


# ============================================================
# 35. CÓDIGOS 30 E 40 AO LONGO DOS ANOS
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    'CÓDIGOS 30 E 40 NAS TRÊS POSIÇÕES'
)

print(
    '=' * 90
)


display(

    df_codigos_consolidado.loc[
        df_codigos_consolidado[
            'Eh_Doenca_30_40'
        ]
        ==
        True
    ]
    .sort_values(
        [
            'Ano',
            'Codigo'
        ]
    )
)


# ============================================================
# 36. SENTINELAS 99 E 999
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    'SENTINELAS 99 E 999'
)

print(
    '=' * 90
)


display(

    df_codigos_consolidado.loc[
        df_codigos_consolidado[
            'Eh_Sentinela_99_999'
        ]
        ==
        True
    ]
    .sort_values(
        [
            'Ano',
            'Codigo'
        ]
    )
)


# ============================================================
# 37. RESULTADOS SALVOS
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    'RESULTADOS SALVOS EM'
)

print(
    '=' * 90
)


print(
    PASTA_SAIDA
)

In [ ]:
# ============================================================
# ETL E AUDITORIA DAS VARIÁVEIS EXPLICATIVAS
# RAIS - PROFESSORES - 5 FAMÍLIAS CBO
#
# Anos:
#   2020, 2021, 2022, 2023, 2024, 2025
#
# ENTRADA:
#   36 Parquets já filtrados:
#   RAIS_PROFESSORES_5CBO
#
# SAÍDA:
#   Parquets harmonizados em:
#   RAIS_HARMONIZADA_ETL
#
# OBJETIVOS:
#
#   1. harmonizar variáveis entre os dois layouts da RAIS
#
#   2. preservar apenas variáveis potencialmente úteis
#      para o ML e variáveis necessárias para auditoria
#
#   3. construir:
#
#          Y_doenca = 1
#
#      se Causa Afastamento 1, 2 ou 3 estiver em {30, 40}
#
#   4. NÃO utilizar como preditores:
#
#      - Causa Afastamento 1
#      - Causa Afastamento 2
#      - Causa Afastamento 3
#      - Qtd Dias Afastamento
#      - remuneração
#      - datas de afastamento
#      - variáveis claramente posteriores ao desfecho
#
#   5. auditar:
#
#      - disponibilidade das variáveis
#      - valores ausentes
#      - cardinalidade
#      - distribuição das categorias
#      - mudanças de códigos ao longo dos anos
#      - resumo das variáveis numéricas
#
# ============================================================


# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

from google.colab import drive

import os
import re
import glob
import math
import unicodedata

from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

drive.mount(
    '/content/drive',
    force_remount=False
)


# ============================================================
# 3. PASTAS
# ============================================================

PASTA_ENTRADA = (
    '/content/drive/MyDrive/TCC_2/dados/'
    'RAIS_PROFESSORES_5CBO'
)


PASTA_HARMONIZADA = (
    '/content/drive/MyDrive/TCC_2/dados/'
    'RAIS_HARMONIZADA_ETL'
)


PASTA_AUDITORIA = (
    '/content/drive/MyDrive/TCC_2/resultados/'
    'AUDITORIA_VARIAVEIS_EXPLICATIVAS'
)


os.makedirs(
    PASTA_HARMONIZADA,
    exist_ok=True
)

os.makedirs(
    PASTA_AUDITORIA,
    exist_ok=True
)


# ============================================================
# 4. CONFIGURAÇÕES
# ============================================================

ANOS = [
    2020,
    2021,
    2022,
    2023,
    2024,
    2025
]


BATCH_SIZE = 250_000


# ------------------------------------------------------------
# TRUE:
# remove eventual Parquet harmonizado anterior
# e refaz o processamento.
#
# Para esta primeira execução, recomendo TRUE.
# ------------------------------------------------------------

SOBRESCREVER_HARMONIZADOS = True


# ============================================================
# 5. POPULAÇÃO CBO
# ============================================================

MAPA_NIVEL_ENSINO = {

    '2312':
        'Fundamental - anos iniciais - nível superior',

    '2313':
        'Fundamental - anos finais - nível superior',

    '2321':
        'Ensino Médio',

    '3312':
        'Fundamental - nível médio',

    '3321':
        'Fundamental - professor leigo'
}


# ============================================================
# 6. NATUREZA JURÍDICA - MACROCATEGORIAS
#
# A primeira posição do código de Natureza Jurídica
# identifica a grande categoria.
#
# Isso é muito mais estável para comparação temporal
# do que utilizar diretamente códigos como 1031 ou 1244.
# ============================================================

MAPA_NATUREZA_MACRO = {

    '1':
        'Administração Pública',

    '2':
        'Entidades Empresariais',

    '3':
        'Entidades sem Fins Lucrativos',

    '4':
        'Pessoas Físicas',

    '5':
        'Organizações Internacionais / Extraterritoriais'
}


# ============================================================
# 7. CÓDIGOS DO DESFECHO
# ============================================================

CODIGOS_DOENCA = {
    30,
    40
}


# ============================================================
# 8. NORMALIZAÇÃO DE TEXTO
# ============================================================

def normalizar_texto(texto):

    texto = str(
        texto
    )

    texto = unicodedata.normalize(
        'NFKD',
        texto
    )

    texto = ''.join(

        caractere

        for caractere in texto

        if not unicodedata.combining(
            caractere
        )
    )

    texto = texto.upper()

    texto = re.sub(
        r'[^A-Z0-9]+',
        ' ',
        texto
    )

    texto = re.sub(
        r'\s+',
        ' ',
        texto
    ).strip()

    return texto


# ============================================================
# 9. PADRONIZAR CATEGORIA
# ============================================================

def padronizar_categoria(serie):

    s = (
        serie
        .astype('string')
        .str.strip()
    )


    # --------------------------------------------------------
    # Ausências textuais
    # --------------------------------------------------------

    s = s.mask(

        s.str.upper().isin(
            [
                '',
                'NAN',
                'NONE',
                '<NA>',
                'NULL'
            ]
        )
    )


    # --------------------------------------------------------
    # Exemplo:
    #
    # 1.0 -> 1
    # 10.0 -> 10
    #
    # Sem destruir zeros à esquerda:
    #
    # 01113 permanece 01113
    # --------------------------------------------------------

    s = s.str.replace(

        r'^([0-9]+)[\.,]0+$',

        r'\1',

        regex=True
    )


    return s.astype(
        'string'
    )


# ============================================================
# 10. PADRONIZAR VARIÁVEL NUMÉRICA
# ============================================================

def padronizar_numerica(serie):

    s = (

        serie
        .astype('string')
        .str.strip()
        .str.replace(
            ',',
            '.',
            regex=False
        )
    )


    return pd.to_numeric(
        s,
        errors='coerce'
    )


# ============================================================
# 11. LIMPAR CÓDIGO DE CAUSA
# ============================================================

def limpar_codigo_causa(serie):

    numero = padronizar_numerica(
        serie
    )


    inteiro = (

        numero.isna()

        |

        np.isclose(
            numero,
            numero.round(),
            equal_nan=True
        )
    )


    numero = numero.where(
        inteiro
    )


    return (
        numero
        .round()
        .astype('Int64')
    )


# ============================================================
# 12. EXTRAIR APENAS DÍGITOS
# ============================================================

def somente_digitos(serie):

    s = padronizar_categoria(
        serie
    )


    return (

        s
        .str.replace(
            r'[^0-9]',
            '',
            regex=True
        )

        .replace(
            '',
            pd.NA
        )

        .astype(
            'string'
        )
    )


# ============================================================
# 13. IDENTIFICAR ANO E GRUPO PELO NOME
# ============================================================

def identificar_arquivo(caminho):

    nome = os.path.basename(
        caminho
    )


    padrao = (
        r'RAIS_PROFESSORES_5CBO_'
        r'(\d{4})_(.+)\.parquet$'
    )


    resultado = re.search(
        padrao,
        nome,
        flags=re.IGNORECASE
    )


    if resultado is None:

        raise RuntimeError(
            f'Nome não reconhecido: {nome}'
        )


    ano = int(
        resultado.group(1)
    )


    grupo = (
        resultado
        .group(2)
        .upper()
    )


    return ano, grupo


# ============================================================
# 14. LOCALIZADOR GENÉRICO DE COLUNAS
# ============================================================

def localizar_coluna(

    colunas,

    aliases=None,

    contem_todos=None,

    contem_algum=None,

    excluir=None,

    preferir=None
):

    aliases = aliases or []

    contem_todos = contem_todos or []

    contem_algum = contem_algum or []

    excluir = excluir or []

    preferir = preferir or []


    mapa = {

        coluna:
            normalizar_texto(
                coluna
            )

        for coluna in colunas
    }


    aliases_norm = {

        normalizar_texto(
            alias
        )

        for alias in aliases
    }


    # --------------------------------------------------------
    # 1. Correspondência exata
    # --------------------------------------------------------

    candidatos = [

        coluna

        for coluna, nome in mapa.items()

        if nome in aliases_norm
    ]


    # --------------------------------------------------------
    # 2. Correspondência semântica
    # --------------------------------------------------------

    if len(
        candidatos
    ) == 0:

        for coluna, nome in mapa.items():

            if not all(

                termo in nome

                for termo
                in contem_todos
            ):

                continue


            if (

                len(
                    contem_algum
                ) > 0

                and

                not any(

                    termo in nome

                    for termo
                    in contem_algum
                )
            ):

                continue


            if any(

                termo in nome

                for termo
                in excluir
            ):

                continue


            candidatos.append(
                coluna
            )


    # --------------------------------------------------------
    # 3. Preferências
    # --------------------------------------------------------

    if (

        len(
            candidatos
        ) > 1

        and

        len(
            preferir
        ) > 0
    ):

        preferidos = [

            coluna

            for coluna
            in candidatos

            if any(

                termo
                in mapa[coluna]

                for termo
                in preferir
            )
        ]


        if len(
            preferidos
        ) > 0:

            candidatos = (
                preferidos
            )


    # --------------------------------------------------------
    # Nada encontrado
    # --------------------------------------------------------

    if len(
        candidatos
    ) == 0:

        return None


    # --------------------------------------------------------
    # Se ainda houver mais de um,
    # escolher o nome mais curto.
    #
    # A decisão ficará registrada na auditoria.
    # --------------------------------------------------------

    candidatos = sorted(

        candidatos,

        key=lambda coluna:
            len(
                mapa[coluna]
            )
    )


    return candidatos[0]


# ============================================================
# 15. IDENTIFICAR CAUSAS 1, 2 E 3
# ============================================================

def localizar_causa(
    colunas,
    numero
):

    mapa = {

        coluna:
            normalizar_texto(
                coluna
            )

        for coluna in colunas
    }


    candidatos = []


    for coluna, nome in mapa.items():

        if (

            'CAUSA'
            in nome

            and

            'AFASTAMENTO'
            in nome

            and

            re.search(
                rf'\b{numero}\b',
                nome
            )
        ):

            candidatos.append(
                coluna
            )


    if len(
        candidatos
    ) > 1:

        codigo = [

            coluna

            for coluna
            in candidatos

            if (
                'CODIGO'
                in mapa[coluna]
            )
        ]


        if len(
            codigo
        ) == 1:

            candidatos = codigo


    if len(
        candidatos
    ) != 1:

        raise RuntimeError(

            f'Não consegui identificar '
            f'Causa Afastamento {numero}. '

            f'Candidatos: {candidatos}'
        )


    return candidatos[0]


# ============================================================
# 16. ESPECIFICAÇÕES DAS VARIÁVEIS
# ============================================================

ESPECIFICACOES = {

    # --------------------------------------------------------
    # IDADE
    # --------------------------------------------------------

    'Idade': {

        'aliases': [
            'Idade'
        ],

        'contem_todos': [
            'IDADE'
        ],

        'excluir': [
            'FAIXA'
        ]
    },


    # --------------------------------------------------------
    # SEXO
    # --------------------------------------------------------

    'Sexo_codigo': {

        'aliases': [
            'Sexo Trabalhador',
            'Sexo - Código',
            'Sexo Código'
        ],

        'contem_todos': [
            'SEXO'
        ],

        'excluir': [
            'ESTAB'
        ],

        'preferir': [
            'CODIGO',
            'TRABALHADOR'
        ]
    },


    # --------------------------------------------------------
    # RAÇA/COR
    # --------------------------------------------------------

    'Raca_cor_codigo': {

        'aliases': [
            'Raça Cor',
            'Raça Cor - Código',
            'Raça Cor Código'
        ],

        'contem_todos': [
            'RACA',
            'COR'
        ],

        'preferir': [
            'CODIGO'
        ]
    },


    # --------------------------------------------------------
    # ESCOLARIDADE
    # --------------------------------------------------------

    'Escolaridade_codigo': {

        'aliases': [
            'Escolaridade após 2005',
            'Escolaridade Após 2005 - Código',
            'Escolaridade após 2005 - Código'
        ],

        'contem_todos': [
            'ESCOLARIDADE'
        ],

        'preferir': [
            'APOS 2005',
            'CODIGO'
        ]
    },


    # --------------------------------------------------------
    # HORAS CONTRATUAIS
    # --------------------------------------------------------

    'Qtd_horas_contratuais': {

        'aliases': [
            'Qtd Hora Contr'
        ],

        'contem_todos': [
            'QTD',
            'HORA',
            'CONTR'
        ]
    },


    # --------------------------------------------------------
    # TEMPO DE EMPREGO
    # --------------------------------------------------------

    'Tempo_emprego': {

        'aliases': [
            'Tempo Emprego'
        ],

        'contem_todos': [
            'TEMPO',
            'EMPREGO'
        ]
    },


    # --------------------------------------------------------
    # TIPO DE VÍNCULO
    # --------------------------------------------------------

    'Tipo_vinculo_codigo': {

        'aliases': [
            'Tipo Vínculo',
            'Tipo Vínculo - Código',
            'Tipo Vinculo - Codigo'
        ],

        'contem_todos': [
            'TIPO',
            'VINCULO'
        ],

        'preferir': [
            'CODIGO'
        ]
    },


    # --------------------------------------------------------
    # NATUREZA JURÍDICA
    # --------------------------------------------------------

    'Natureza_juridica_codigo': {

        'aliases': [
            'Natureza Jurídica',
            'Natureza Jurídica - Código'
        ],

        'contem_todos': [
            'NATUREZA',
            'JURIDICA'
        ],

        'preferir': [
            'CODIGO'
        ]
    },


    # --------------------------------------------------------
    # CNAE
    # --------------------------------------------------------

    'CNAE_classe_codigo': {

        'aliases': [
            'CNAE 2.0 Classe',
            'CNAE 2.0 Classe - Código'
        ],

        'contem_todos': [
            'CNAE',
            'CLASSE'
        ],

        'preferir': [
            'CODIGO'
        ]
    },


    # --------------------------------------------------------
    # TAMANHO DO ESTABELECIMENTO
    # --------------------------------------------------------

    'Tamanho_estabelecimento_codigo': {

        'aliases': [
            'Tamanho Estabelecimento',
            'Tamanho Estabelecimento - Código'
        ],

        'contem_todos': [
            'TAMANHO',
            'ESTABELECIMENTO'
        ],

        'preferir': [
            'CODIGO'
        ]
    },


    # --------------------------------------------------------
    # DEFICIÊNCIA
    # --------------------------------------------------------

    'Indicador_deficiencia_codigo': {

        'aliases': [
            'Ind Portador Defic',
            'Ind Portador Deficiência',
            'Ind Portador Deficiência - Código'
        ],

        'contem_todos': [
            'DEFIC'
        ],

        'contem_algum': [
            'IND',
            'PORTADOR'
        ],

        'preferir': [
            'CODIGO',
            'IND'
        ]
    },


    # --------------------------------------------------------
    # TIPO DE ADMISSÃO
    # --------------------------------------------------------

    'Tipo_admissao_codigo': {

        'aliases': [
            'Tipo Admissão',
            'Tipo Admissão - Código'
        ],

        'contem_todos': [
            'TIPO',
            'ADMIS'
        ],

        'preferir': [
            'CODIGO'
        ]
    }
}


# ============================================================
# 17. IDENTIFICAR LAYOUT DAS VARIÁVEIS
# ============================================================

def identificar_layout(
    colunas
):

    layout = {}


    # --------------------------------------------------------
    # Causas
    # --------------------------------------------------------

    layout[
        'Causa_1'
    ] = localizar_causa(
        colunas,
        1
    )


    layout[
        'Causa_2'
    ] = localizar_causa(
        colunas,
        2
    )


    layout[
        'Causa_3'
    ] = localizar_causa(
        colunas,
        3
    )


    # --------------------------------------------------------
    # Variáveis explicativas
    # --------------------------------------------------------

    for variavel, especificacao in (
        ESPECIFICACOES.items()
    ):

        layout[
            variavel
        ] = localizar_coluna(

            colunas,

            aliases=
                especificacao.get(
                    'aliases'
                ),

            contem_todos=
                especificacao.get(
                    'contem_todos'
                ),

            contem_algum=
                especificacao.get(
                    'contem_algum'
                ),

            excluir=
                especificacao.get(
                    'excluir'
                ),

            preferir=
                especificacao.get(
                    'preferir'
                )
        )


    return layout


# ============================================================
# 18. CRIAR SÉRIE VAZIA
# ============================================================

def serie_string_vazia(
    indice
):

    return pd.Series(

        pd.NA,

        index=indice,

        dtype='string'
    )


def serie_numerica_vazia(
    indice
):

    return pd.Series(

        np.nan,

        index=indice,

        dtype='float32'
    )


# ============================================================
# 19. LER CATEGORIA OPCIONAL
# ============================================================

def obter_categoria(
    df,
    coluna
):

    if coluna is None:

        return serie_string_vazia(
            df.index
        )


    return padronizar_categoria(
        df[coluna]
    )


# ============================================================
# 20. LER NUMÉRICA OPCIONAL
# ============================================================

def obter_numerica(
    df,
    coluna
):

    if coluna is None:

        return serie_numerica_vazia(
            df.index
        )


    return (

        padronizar_numerica(
            df[coluna]
        )

        .astype(
            'float32'
        )
    )


# ============================================================
# 21. VARIÁVEIS NUMÉRICAS AUDITADAS
# ============================================================

VARIAVEIS_NUMERICAS = [

    'Idade',

    'Qtd_horas_contratuais',

    'Tempo_emprego'
]


# ============================================================
# 22. VARIÁVEIS CATEGÓRICAS AUDITADAS
# ============================================================

VARIAVEIS_CATEGORICAS = [

    'UF',

    'Familia_CBO',

    'CBO_6',

    'Sexo_codigo',

    'Raca_cor_codigo',

    'Escolaridade_codigo',

    'Tipo_vinculo_codigo',

    'Natureza_juridica_codigo',

    'Natureza_macro',

    'Setor_publico',

    'CNAE_classe_codigo',

    'CNAE_divisao',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo',

    'Tipo_admissao_codigo',

    'Municipio_codigo'
]


# ------------------------------------------------------------
# Para distribuição completa não precisamos imprimir
# milhares de municípios.
# ------------------------------------------------------------

VARIAVEIS_DISTRIBUICAO = [

    'UF',

    'Familia_CBO',

    'CBO_6',

    'Sexo_codigo',

    'Raca_cor_codigo',

    'Escolaridade_codigo',

    'Tipo_vinculo_codigo',

    'Natureza_juridica_codigo',

    'Natureza_macro',

    'Setor_publico',

    'CNAE_divisao',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo',

    'Tipo_admissao_codigo'
]


# ============================================================
# 23. LOCALIZAR PARQUETS
# ============================================================

arquivos = sorted(

    glob.glob(

        os.path.join(
            PASTA_ENTRADA,
            '**',
            '*.parquet'
        ),

        recursive=True
    )
)


print(
    f'Parquets encontrados: '
    f'{len(arquivos)}'
)


if len(
    arquivos
) != 36:

    print(
        '\nATENÇÃO: eram esperados 36 arquivos.'
    )


# ============================================================
# 24. ACUMULADORES DA AUDITORIA
# ============================================================

total_ano = Counter()

doenca_ano = Counter()


# ------------------------------------------------------------
# Missing
# ------------------------------------------------------------

total_variavel = Counter()

missing_variavel = Counter()


# ------------------------------------------------------------
# Cardinalidade
# ------------------------------------------------------------

valores_unicos = defaultdict(
    set
)


# ------------------------------------------------------------
# Distribuições categóricas
# ------------------------------------------------------------

distribuicao = Counter()


# ------------------------------------------------------------
# Estatísticas numéricas
# ------------------------------------------------------------

estatisticas_numericas = defaultdict(
    lambda: {
        'n': 0,
        'soma': 0.0,
        'soma_quadrados': 0.0,
        'min': None,
        'max': None
    }
)


# ------------------------------------------------------------
# Disponibilidade de colunas
# ------------------------------------------------------------

mapa_colunas = []


# ------------------------------------------------------------
# Status
# ------------------------------------------------------------

status_processamento = []


# ============================================================
# 25. ATUALIZAR AUDITORIA
# ============================================================

def atualizar_auditoria(
    base,
    ano
):

    quantidade = len(
        base
    )


    total_ano[
        ano
    ] += quantidade


    doenca_ano[
        ano
    ] += int(

        base[
            'Y_doenca'
        ]
        .fillna(0)
        .sum()
    )


    # ========================================================
    # TODAS AS VARIÁVEIS
    # ========================================================

    todas_variaveis = (

        VARIAVEIS_NUMERICAS

        +

        VARIAVEIS_CATEGORICAS
    )


    for variavel in todas_variaveis:

        serie = base[
            variavel
        ]


        total_variavel[
            (
                ano,
                variavel
            )
        ] += quantidade


        faltantes = int(
            serie.isna().sum()
        )


        missing_variavel[
            (
                ano,
                variavel
            )
        ] += faltantes


    # ========================================================
    # CATEGÓRICAS
    # ========================================================

    for variavel in (
        VARIAVEIS_CATEGORICAS
    ):

        serie = (

            base[
                variavel
            ]

            .astype(
                'string'
            )
        )


        validos = (
            serie
            .dropna()
        )


        valores_unicos[
            (
                ano,
                variavel
            )
        ].update(

            validos.unique().tolist()
        )


        # ----------------------------------------------------
        # Distribuições somente das selecionadas
        # ----------------------------------------------------

        if (
            variavel
            in VARIAVEIS_DISTRIBUICAO
        ):

            contagens = (

                serie
                .fillna(
                    '<NA>'
                )
                .value_counts(
                    dropna=False
                )
            )


            for valor, qtd in (
                contagens.items()
            ):

                distribuicao[
                    (
                        ano,
                        variavel,
                        str(valor)
                    )
                ] += int(
                    qtd
                )


    # ========================================================
    # NUMÉRICAS
    # ========================================================

    for variavel in (
        VARIAVEIS_NUMERICAS
    ):

        serie = (

            pd.to_numeric(
                base[
                    variavel
                ],
                errors='coerce'
            )

            .dropna()
        )


        if len(
            serie
        ) == 0:

            continue


        chave = (
            ano,
            variavel
        )


        estat = (
            estatisticas_numericas[
                chave
            ]
        )


        valores = (
            serie.astype(
                'float64'
            )
        )


        estat[
            'n'
        ] += len(
            valores
        )


        estat[
            'soma'
        ] += float(
            valores.sum()
        )


        estat[
            'soma_quadrados'
        ] += float(
            np.square(
                valores
            ).sum()
        )


        minimo = float(
            valores.min()
        )


        maximo = float(
            valores.max()
        )


        if (

            estat[
                'min'
            ]
            is None

            or

            minimo
            <
            estat[
                'min'
            ]
        ):

            estat[
                'min'
            ] = minimo


        if (

            estat[
                'max'
            ]
            is None

            or

            maximo
            >
            estat[
                'max'
            ]
        ):

            estat[
                'max'
            ] = maximo


# ============================================================
# 26. PROCESSAR CADA PARQUET
# ============================================================

for indice_arquivo, arquivo in enumerate(

    arquivos,

    start=1
):

    print(
        '\n'
        +
        '=' * 90
    )


    ano, grupo = identificar_arquivo(
        arquivo
    )


    print(

        f'[{indice_arquivo}/'
        f'{len(arquivos)}] '

        f'{ano} - {grupo}'
    )


    parquet = pq.ParquetFile(
        arquivo
    )


    colunas_origem = (
        parquet.schema.names
    )


    layout = identificar_layout(
        colunas_origem
    )


    # ========================================================
    # MAPEAR COLUNAS
    # ========================================================

    for variavel, coluna_origem in (
        layout.items()
    ):

        mapa_colunas.append({

            'Ano':
                ano,

            'Grupo':
                grupo,

            'Variavel_harmonizada':
                variavel,

            'Coluna_origem':
                coluna_origem,

            'Presente':
                coluna_origem is not None
        })


    # ========================================================
    # COLUNAS FIXAS JÁ CRIADAS NA ETAPA ANTERIOR
    # ========================================================

    for variavel in [

        'UF',
        'Familia_CBO',
        'CBO_padronizada',
        'Municipio_padronizado',
        'Grupo_Origem'
    ]:

        mapa_colunas.append({

            'Ano':
                ano,

            'Grupo':
                grupo,

            'Variavel_harmonizada':
                variavel,

            'Coluna_origem':
                (
                    variavel

                    if variavel
                    in colunas_origem

                    else None
                ),

            'Presente':
                variavel
                in colunas_origem
        })


    # ========================================================
    # COLUNAS QUE PRECISAMOS LER
    # ========================================================

    colunas_leitura = set()


    # --------------------------------------------------------
    # Obrigatórias
    # --------------------------------------------------------

    for coluna in [

        'UF',

        'Familia_CBO',

        'CBO_padronizada',

        'Municipio_padronizado'
    ]:

        if coluna not in colunas_origem:

            raise RuntimeError(

                f'{arquivo}: '
                f'coluna obrigatória ausente: '
                f'{coluna}'
            )


        colunas_leitura.add(
            coluna
        )


    # --------------------------------------------------------
    # Causas
    # --------------------------------------------------------

    for chave in [

        'Causa_1',
        'Causa_2',
        'Causa_3'
    ]:

        colunas_leitura.add(
            layout[chave]
        )


    # --------------------------------------------------------
    # Opcionais
    # --------------------------------------------------------

    for variavel in (
        ESPECIFICACOES.keys()
    ):

        coluna = (
            layout[
                variavel
            ]
        )


        if coluna is not None:

            colunas_leitura.add(
                coluna
            )


    colunas_leitura = list(
        colunas_leitura
    )


    # ========================================================
    # ARQUIVO HARMONIZADO DE SAÍDA
    # ========================================================

    pasta_ano_saida = os.path.join(

        PASTA_HARMONIZADA,

        str(
            ano
        )
    )


    os.makedirs(
        pasta_ano_saida,
        exist_ok=True
    )


    saida = os.path.join(

        pasta_ano_saida,

        (
            f'RAIS_HARMONIZADA_'
            f'{ano}_{grupo}.parquet'
        )
    )


    if (

        SOBRESCREVER_HARMONIZADOS

        and

        os.path.exists(
            saida
        )
    ):

        os.remove(
            saida
        )


    writer = None

    linhas_processadas = 0

    casos_doenca = 0


    try:

        # ====================================================
        # BATCHES
        # ====================================================

        for numero_batch, batch in enumerate(

            parquet.iter_batches(

                batch_size=
                    BATCH_SIZE,

                columns=
                    colunas_leitura
            ),

            start=1
        ):

            df = batch.to_pandas()


            linhas_processadas += len(
                df
            )


            # =================================================
            # TARGET
            # =================================================

            causa1 = limpar_codigo_causa(

                df[
                    layout[
                        'Causa_1'
                    ]
                ]
            )


            causa2 = limpar_codigo_causa(

                df[
                    layout[
                        'Causa_2'
                    ]
                ]
            )


            causa3 = limpar_codigo_causa(

                df[
                    layout[
                        'Causa_3'
                    ]
                ]
            )


            y_doenca = (

                causa1.isin(
                    CODIGOS_DOENCA
                )

                |

                causa2.isin(
                    CODIGOS_DOENCA
                )

                |

                causa3.isin(
                    CODIGOS_DOENCA
                )
            )


            casos_doenca += int(
                y_doenca.sum()
            )


            # =================================================
            # IDENTIFICAÇÃO
            # =================================================

            uf = (

                df[
                    'UF'
                ]

                .astype(
                    'string'
                )

                .str.strip()

                .str.upper()
            )


            familia = (

                df[
                    'Familia_CBO'
                ]

                .astype(
                    'string'
                )

                .str.extract(
                    r'(\d{4})',
                    expand=False
                )
            )


            cbo6 = (

                df[
                    'CBO_padronizada'
                ]

                .astype(
                    'string'
                )

                .str.extract(
                    r'(\d{6})',
                    expand=False
                )
            )


            municipio = (

                df[
                    'Municipio_padronizado'
                ]

                .astype(
                    'string'
                )

                .str.extract(
                    r'(\d{6,7})',
                    expand=False
                )
            )


            # =================================================
            # VARIÁVEIS NUMÉRICAS
            # =================================================

            idade = obter_numerica(

                df,

                layout[
                    'Idade'
                ]
            )


            horas = obter_numerica(

                df,

                layout[
                    'Qtd_horas_contratuais'
                ]
            )


            tempo_emprego = obter_numerica(

                df,

                layout[
                    'Tempo_emprego'
                ]
            )


            # =================================================
            # VARIÁVEIS CATEGÓRICAS
            # =================================================

            sexo = obter_categoria(

                df,

                layout[
                    'Sexo_codigo'
                ]
            )


            raca = obter_categoria(

                df,

                layout[
                    'Raca_cor_codigo'
                ]
            )


            escolaridade = obter_categoria(

                df,

                layout[
                    'Escolaridade_codigo'
                ]
            )


            tipo_vinculo = obter_categoria(

                df,

                layout[
                    'Tipo_vinculo_codigo'
                ]
            )


            natureza = obter_categoria(

                df,

                layout[
                    'Natureza_juridica_codigo'
                ]
            )


            cnae = obter_categoria(

                df,

                layout[
                    'CNAE_classe_codigo'
                ]
            )


            tamanho_estab = obter_categoria(

                df,

                layout[
                    'Tamanho_estabelecimento_codigo'
                ]
            )


            deficiencia = obter_categoria(

                df,

                layout[
                    'Indicador_deficiencia_codigo'
                ]
            )


            tipo_admissao = obter_categoria(

                df,

                layout[
                    'Tipo_admissao_codigo'
                ]
            )


            # =================================================
            # NATUREZA JURÍDICA HARMONIZADA
            # =================================================

            natureza_digitos = (
                somente_digitos(
                    natureza
                )
            )


            prefixo_natureza = (

                natureza_digitos

                .str[:1]
            )


            natureza_macro = (

                prefixo_natureza

                .map(
                    MAPA_NATUREZA_MACRO
                )

                .astype(
                    'string'
                )
            )


            # -------------------------------------------------
            # SETOR PÚBLICO
            # -------------------------------------------------

            setor_publico = pd.Series(

                pd.NA,

                index=df.index,

                dtype='Int8'
            )


            mascara_natureza = (
                prefixo_natureza
                .notna()
            )


            setor_publico.loc[
                mascara_natureza
            ] = (

                prefixo_natureza.loc[
                    mascara_natureza
                ]
                .eq(
                    '1'
                )
                .astype(
                    'int8'
                )
            )


            # =================================================
            # CNAE HARMONIZADA
            # =================================================

            cnae_digitos = (

                somente_digitos(
                    cnae
                )
            )


            cnae_divisao = (

                cnae_digitos

                .str[:2]

                .where(
                    cnae_digitos
                    .str.len()
                    >=
                    2
                )

                .astype(
                    'string'
                )
            )


            # =================================================
            # NÍVEL DE ENSINO
            # =================================================

            nivel_ensino = (

                familia

                .map(
                    MAPA_NIVEL_ENSINO
                )

                .astype(
                    'string'
                )
            )


            # =================================================
            # BASE HARMONIZADA
            #
            # NÃO entram:
            #
            # - causas de afastamento
            # - dias de afastamento
            # - remunerações
            # - datas de afastamento
            #
            # =================================================

            base = pd.DataFrame({

                'Ano':
                    pd.Series(
                        ano,
                        index=df.index,
                        dtype='Int16'
                    ),

                'UF':
                    uf,

                'Grupo_Origem':
                    pd.Series(
                        grupo,
                        index=df.index,
                        dtype='string'
                    ),

                'Municipio_codigo':
                    municipio,

                'CBO_6':
                    cbo6,

                'Familia_CBO':
                    familia,

                'Nivel_Ensino':
                    nivel_ensino,

                # ---------------------------------------------
                # TARGET
                # ---------------------------------------------

                'Y_doenca':
                    y_doenca.astype(
                        'Int8'
                    ),

                # ---------------------------------------------
                # PREDITORES CANDIDATOS
                # ---------------------------------------------

                'Idade':
                    idade.astype(
                        'float32'
                    ),

                'Sexo_codigo':
                    sexo,

                'Raca_cor_codigo':
                    raca,

                'Escolaridade_codigo':
                    escolaridade,

                'Qtd_horas_contratuais':
                    horas.astype(
                        'float32'
                    ),

                'Tempo_emprego':
                    tempo_emprego.astype(
                        'float32'
                    ),

                'Tipo_vinculo_codigo':
                    tipo_vinculo,

                # ---------------------------------------------
                # Natureza bruta fica para auditoria.
                #
                # Para o modelo principal, a tendência será
                # usar Natureza_macro / Setor_publico.
                # ---------------------------------------------

                'Natureza_juridica_codigo':
                    natureza,

                'Natureza_macro':
                    natureza_macro,

                'Setor_publico':
                    setor_publico,

                # ---------------------------------------------
                # CNAE
                # ---------------------------------------------

                'CNAE_classe_codigo':
                    cnae,

                'CNAE_divisao':
                    cnae_divisao,

                # ---------------------------------------------
                # Demais candidatas
                # ---------------------------------------------

                'Tamanho_estabelecimento_codigo':
                    tamanho_estab,

                'Indicador_deficiencia_codigo':
                    deficiencia,

                'Tipo_admissao_codigo':
                    tipo_admissao
            })


            # =================================================
            # AUDITORIA
            # =================================================

            atualizar_auditoria(
                base,
                ano
            )


            # =================================================
            # SALVAR PARQUET
            # =================================================

            tabela = pa.Table.from_pandas(

                base,

                preserve_index=False
            )


            if writer is None:

                writer = pq.ParquetWriter(

                    saida,

                    tabela.schema,

                    compression='snappy'
                )


            writer.write_table(
                tabela
            )


            print(

                f'Batch {numero_batch}: '

                f'{linhas_processadas:,} '
                f'linhas acumuladas | '

                f'Y=1 acumulado: '
                f'{casos_doenca:,}'
            )


        # ====================================================
        # FECHAR WRITER
        # ====================================================

        if writer is not None:

            writer.close()

            writer = None


        # ====================================================
        # VALIDAR SAÍDA
        # ====================================================

        parquet_saida = pq.ParquetFile(
            saida
        )


        linhas_saida = (
            parquet_saida
            .metadata
            .num_rows
        )


        if (
            linhas_saida
            !=
            linhas_processadas
        ):

            raise RuntimeError(

                f'Divergência no arquivo '
                f'{saida}: '

                f'{linhas_processadas:,} processadas '

                f'e {linhas_saida:,} salvas.'
            )


        prevalencia = (

            casos_doenca
            /
            linhas_processadas
            *
            100
        )


        status_processamento.append({

            'Ano':
                ano,

            'Grupo':
                grupo,

            'Arquivo_origem':
                os.path.basename(
                    arquivo
                ),

            'Arquivo_saida':
                os.path.basename(
                    saida
                ),

            'Linhas':
                linhas_processadas,

            'Doenca':
                casos_doenca,

            'Pct_doenca':
                prevalencia,

            'Status':
                'OK'
        })


        print(
            '\nArquivo concluído:'
        )

        print(
            saida
        )

        print(

            f'Linhas: '
            f'{linhas_processadas:,}'
        )

        print(

            f'Y_doenca = 1: '
            f'{casos_doenca:,} '
            f'({prevalencia:.3f}%)'
        )


    except Exception as erro:

        if writer is not None:

            writer.close()


        if os.path.exists(
            saida
        ):

            try:

                os.remove(
                    saida
                )

            except Exception:

                pass


        status_processamento.append({

            'Ano':
                ano,

            'Grupo':
                grupo,

            'Arquivo_origem':
                os.path.basename(
                    arquivo
                ),

            'Arquivo_saida':
                os.path.basename(
                    saida
                ),

            'Linhas':
                None,

            'Doenca':
                None,

            'Pct_doenca':
                None,

            'Status':
                str(
                    erro
                )
        })


        print(
            f'ERRO: {erro}'
        )


# ============================================================
# 27. STATUS DO PROCESSAMENTO
# ============================================================

df_status = pd.DataFrame(
    status_processamento
)


df_status.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '01_status_processamento.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 28. MAPA DAS COLUNAS
# ============================================================

df_mapa = pd.DataFrame(
    mapa_colunas
)


df_mapa.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '02_mapa_colunas_por_arquivo.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 29. DISPONIBILIDADE POR ANO
# ============================================================

df_disponibilidade = (

    df_mapa

    .groupby(
        [
            'Ano',
            'Variavel_harmonizada'
        ],
        as_index=False
    )

    .agg(

        Arquivos=(
            'Grupo',
            'count'
        ),

        Arquivos_com_variavel=(
            'Presente',
            'sum'
        )
    )
)


df_disponibilidade[
    'Pct_arquivos_com_variavel'
] = (

    df_disponibilidade[
        'Arquivos_com_variavel'
    ]

    /

    df_disponibilidade[
        'Arquivos'
    ]

    * 100
)


df_disponibilidade.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '03_disponibilidade_variaveis_por_ano.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 30. MISSINGNESS POR ANO
# ============================================================

registros = []


for (
    ano,
    variavel
), total in (
    total_variavel.items()
):

    faltantes = (

        missing_variavel[
            (
                ano,
                variavel
            )
        ]
    )


    registros.append({

        'Ano':
            ano,

        'Variavel':
            variavel,

        'N':
            total,

        'Ausentes':
            faltantes,

        'Validos':
            total
            -
            faltantes,

        'Pct_ausentes':
            (
                faltantes
                /
                total
                *
                100
            )
            if total > 0
            else np.nan
    })


df_missing = pd.DataFrame(
    registros
)


df_missing = (

    df_missing

    .sort_values(
        [
            'Variavel',
            'Ano'
        ]
    )

    .reset_index(
        drop=True
    )
)


df_missing.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '04_missingness_por_ano.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 31. CARDINALIDADE
# ============================================================

registros = []


for (
    ano,
    variavel
), valores in (
    valores_unicos.items()
):

    registros.append({

        'Ano':
            ano,

        'Variavel':
            variavel,

        'Qtd_valores_distintos':
            len(
                valores
            )
    })


df_cardinalidade = pd.DataFrame(
    registros
)


df_cardinalidade = (

    df_cardinalidade

    .sort_values(
        [
            'Variavel',
            'Ano'
        ]
    )

    .reset_index(
        drop=True
    )
)


df_cardinalidade.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '05_cardinalidade_por_ano.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 32. DISTRIBUIÇÃO DAS CATEGÓRICAS
# ============================================================

registros = []


for (
    ano,
    variavel,
    valor
), quantidade in (
    distribuicao.items()
):

    registros.append({

        'Ano':
            ano,

        'Variavel':
            variavel,

        'Valor':
            valor,

        'Quantidade':
            quantidade
    })


df_distribuicao = pd.DataFrame(
    registros
)


if len(
    df_distribuicao
) > 0:

    df_distribuicao[
        'Total_variavel_ano'
    ] = (

        df_distribuicao

        .groupby(
            [
                'Ano',
                'Variavel'
            ]
        )[
            'Quantidade'
        ]

        .transform(
            'sum'
        )
    )


    df_distribuicao[
        'Percentual'
    ] = (

        df_distribuicao[
            'Quantidade'
        ]

        /

        df_distribuicao[
            'Total_variavel_ano'
        ]

        * 100
    )


    df_distribuicao = (

        df_distribuicao

        .sort_values(
            [
                'Variavel',
                'Ano',
                'Quantidade'
            ],

            ascending=[
                True,
                True,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )


df_distribuicao.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '06_distribuicao_categoricas.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 33. RESUMO NUMÉRICO
# ============================================================

registros = []


for (
    ano,
    variavel
), estat in (
    estatisticas_numericas.items()
):

    n = estat[
        'n'
    ]


    if n > 0:

        media = (
            estat[
                'soma'
            ]
            /
            n
        )


        if n > 1:

            variancia = (

                estat[
                    'soma_quadrados'
                ]

                -

                (
                    estat[
                        'soma'
                    ]
                    ** 2
                    /
                    n
                )
            )

            variancia = max(
                variancia,
                0
            )

            variancia = (
                variancia
                /
                (
                    n
                    -
                    1
                )
            )


            desvio = math.sqrt(
                variancia
            )

        else:

            desvio = np.nan


    else:

        media = np.nan

        desvio = np.nan


    total = total_variavel.get(
        (
            ano,
            variavel
        ),
        0
    )


    faltantes = missing_variavel.get(
        (
            ano,
            variavel
        ),
        0
    )


    registros.append({

        'Ano':
            ano,

        'Variavel':
            variavel,

        'N_total':
            total,

        'N_valido':
            n,

        'Ausentes':
            faltantes,

        'Pct_ausentes':
            (
                faltantes
                /
                total
                *
                100
            )
            if total > 0
            else np.nan,

        'Media':
            media,

        'Desvio_padrao':
            desvio,

        'Minimo':
            estat[
                'min'
            ],

        'Maximo':
            estat[
                'max'
            ]
    })


df_numericas = pd.DataFrame(
    registros
)


df_numericas = (

    df_numericas

    .sort_values(
        [
            'Variavel',
            'Ano'
        ]
    )

    .reset_index(
        drop=True
    )
)


df_numericas.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '07_resumo_numericas_por_ano.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 34. NÍVEIS PRESENTES AO LONGO DOS 6 ANOS
#
# Fundamental para identificar mudanças de códigos.
# ============================================================

if len(
    df_distribuicao
) > 0:

    df_niveis = (

        df_distribuicao.loc[
            df_distribuicao[
                'Valor'
            ]
            !=
            '<NA>'
        ]

        .groupby(
            [
                'Variavel',
                'Valor'
            ],
            as_index=False
        )

        .agg(

            Anos_presentes=(
                'Ano',
                'nunique'
            ),

            Primeiro_ano=(
                'Ano',
                'min'
            ),

            Ultimo_ano=(
                'Ano',
                'max'
            ),

            Quantidade_total=(
                'Quantidade',
                'sum'
            )
        )
    )


    df_niveis[
        'Presente_nos_6_anos'
    ] = (

        df_niveis[
            'Anos_presentes'
        ]
        ==
        6
    )


else:

    df_niveis = pd.DataFrame()


df_niveis.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '08_niveis_categoricos_ao_longo_dos_anos.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 35. TARGET POR ANO
# ============================================================

registros = []


for ano in ANOS:

    n = total_ano[
        ano
    ]


    doenca = doenca_ano[
        ano
    ]


    registros.append({

        'Ano':
            ano,

        'N':
            n,

        'Y_doenca_1':
            doenca,

        'Y_doenca_0':
            n
            -
            doenca,

        'Pct_doenca':
            (
                doenca
                /
                n
                *
                100
            )
            if n > 0
            else np.nan
    })


df_target = pd.DataFrame(
    registros
)


df_target.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '09_target_por_ano.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 36. RECOMENDAÇÃO INICIAL DE USO DAS VARIÁVEIS
#
# Isso NÃO é a decisão final.
#
# A decisão será tomada depois da auditoria produzida acima.
# ============================================================

recomendacoes = [

    {
        'Variavel': 'Ano',
        'Papel': 'Candidata / controle temporal',
        'Uso_inicial': 'AVALIAR',
        'Observacao':
            'Pode capturar mudanças estruturais entre anos. '
            'Comparar modelos com e sem Ano.'
    },

    {
        'Variavel': 'UF',
        'Papel': 'Preditor geográfico',
        'Uso_inicial': 'SIM',
        'Observacao':
            'Baixa cardinalidade e cobertura nacional.'
    },

    {
        'Variavel': 'Municipio_codigo',
        'Papel': 'Geografia detalhada',
        'Uso_inicial': 'NAO NO MODELO BASE',
        'Observacao':
            'Cardinalidade muito alta. Manter para auditorias.'
    },

    {
        'Variavel': 'Familia_CBO',
        'Papel': 'Ocupação / nível docente',
        'Uso_inicial': 'SIM',
        'Observacao':
            'Parte central da definição da população.'
    },

    {
        'Variavel': 'CBO_6',
        'Papel': 'Ocupação detalhada',
        'Uso_inicial': 'AVALIAR',
        'Observacao':
            'Mais detalhada que Família_CBO. Ver cardinalidade.'
    },

    {
        'Variavel': 'Idade',
        'Papel': 'Demográfica',
        'Uso_inicial': 'SIM',
        'Observacao':
            'Variável estrutural.'
    },

    {
        'Variavel': 'Sexo_codigo',
        'Papel': 'Demográfica',
        'Uso_inicial': 'SIM',
        'Observacao':
            'Validar códigos entre layouts.'
    },

    {
        'Variavel': 'Raca_cor_codigo',
        'Papel': 'Demográfica',
        'Uso_inicial': 'SIM',
        'Observacao':
            'Avaliar missing e categorias ignoradas.'
    },

    {
        'Variavel': 'Escolaridade_codigo',
        'Papel': 'Formação',
        'Uso_inicial': 'SIM',
        'Observacao':
            'Validar estabilidade dos códigos.'
    },

    {
        'Variavel': 'Qtd_horas_contratuais',
        'Papel': 'Condição de trabalho',
        'Uso_inicial': 'SIM',
        'Observacao':
            'Verificar limites, zeros e outliers.'
    },

    {
        'Variavel': 'Tempo_emprego',
        'Papel': 'Condição de trabalho',
        'Uso_inicial': 'SIM',
        'Observacao':
            'Verificar unidade e distribuição por ano.'
    },

    {
        'Variavel': 'Tipo_vinculo_codigo',
        'Papel': 'Relação trabalhista',
        'Uso_inicial': 'AVALIAR',
        'Observacao':
            'Verificar compatibilidade entre layouts.'
    },

    {
        'Variavel': 'Natureza_juridica_codigo',
        'Papel': 'Natureza jurídica bruta',
        'Uso_inicial': 'NAO',
        'Observacao':
            'Mudança estrutural de códigos identificada '
            'entre 2022 e 2023.'
    },

    {
        'Variavel': 'Natureza_macro',
        'Papel': 'Natureza jurídica harmonizada',
        'Uso_inicial': 'SIM',
        'Observacao':
            'Macrocategoria comparável entre anos.'
    },

    {
        'Variavel': 'Setor_publico',
        'Papel': 'Natureza jurídica harmonizada',
        'Uso_inicial': 'SIM',
        'Observacao':
            'Indicador simples e temporalmente mais robusto.'
    },

    {
        'Variavel': 'CNAE_classe_codigo',
        'Papel': 'Atividade econômica detalhada',
        'Uso_inicial': 'NAO NO MODELO BASE',
        'Observacao':
            'Preferir divisão CNAE inicialmente.'
    },

    {
        'Variavel': 'CNAE_divisao',
        'Papel': 'Atividade econômica harmonizada',
        'Uso_inicial': 'AVALIAR',
        'Observacao':
            'Menor cardinalidade que a classe.'
    },

    {
        'Variavel': 'Tamanho_estabelecimento_codigo',
        'Papel': 'Características do estabelecimento',
        'Uso_inicial': 'AVALIAR',
        'Observacao':
            'Verificar estabilidade dos códigos.'
    },

    {
        'Variavel': 'Indicador_deficiencia_codigo',
        'Papel': 'Característica individual',
        'Uso_inicial': 'AVALIAR',
        'Observacao':
            'Verificar disponibilidade e missing.'
    },

    {
        'Variavel': 'Tipo_admissao_codigo',
        'Papel': 'Relação de emprego',
        'Uso_inicial': 'AVALIAR',
        'Observacao':
            'Verificar comparabilidade temporal.'
    },

    {
        'Variavel': 'Y_doenca',
        'Papel': 'TARGET',
        'Uso_inicial': 'TARGET',
        'Observacao':
            '1 quando alguma causa de afastamento é 30 ou 40.'
    }
]


df_recomendacoes = pd.DataFrame(
    recomendacoes
)


df_recomendacoes.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '10_recomendacao_inicial_variaveis.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 37. VALIDAÇÕES FINAIS
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    'STATUS DO PROCESSAMENTO'
)

print(
    '=' * 90
)


display(
    df_status
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'TARGET POR ANO'
)

print(
    '=' * 90
)


display(
    df_target
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'DISPONIBILIDADE DAS VARIÁVEIS'
)

print(
    '=' * 90
)


display(
    df_disponibilidade
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'MISSINGNESS'
)

print(
    '=' * 90
)


display(
    df_missing
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'RESUMO DAS VARIÁVEIS NUMÉRICAS'
)

print(
    '=' * 90
)


display(
    df_numericas
)


# ============================================================
# 38. ALERTAS AUTOMÁTICOS
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    'ALERTAS - MAIS DE 20% DE AUSÊNCIA'
)

print(
    '=' * 90
)


df_alerta_missing = (

    df_missing.loc[

        df_missing[
            'Pct_ausentes'
        ]
        >
        20
    ]

    .sort_values(
        [
            'Variavel',
            'Ano'
        ]
    )
)


display(
    df_alerta_missing
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'CATEGORIAS NÃO PRESENTES NOS 6 ANOS'
)

print(
    '=' * 90
)


if len(
    df_niveis
) > 0:

    display(

        df_niveis.loc[
            df_niveis[
                'Presente_nos_6_anos'
            ]
            ==
            False
        ]
        .sort_values(
            [
                'Variavel',
                'Quantidade_total'
            ],
            ascending=[
                True,
                False
            ]
        )
    )


# ============================================================
# 39. CONTAGEM DE ARQUIVOS HARMONIZADOS
# ============================================================

arquivos_ok = (

    df_status[
        'Status'
    ]
    .eq(
        'OK'
    )
    .sum()
)


print(
    '\n'
    +
    '=' * 90
)

print(
    f'ARQUIVOS HARMONIZADOS: '
    f'{arquivos_ok} DE 36'
)

print(
    '=' * 90
)


# ============================================================
# 40. CAMINHOS FINAIS
# ============================================================

print(
    '\nParquets harmonizados:'
)

print(
    PASTA_HARMONIZADA
)


print(
    '\nAuditorias:'
)

print(
    PASTA_AUDITORIA
)

In [ ]:
# ============================================================
# HARMONIZAÇÃO FINAL E VALIDAÇÃO PARA MACHINE LEARNING
# RAIS - PROFESSORES - 2020 A 2025
#
# ENTRADA:
#   /RAIS_HARMONIZADA_ETL/
#
# SAÍDA:
#   /RAIS_BASE_MODELO_FINAL/
#
# OBJETIVOS:
#
# 1. remover diferenças puramente de representação:
#       "01" -> "1"
#       "02" -> "2"
#       "09" -> "9"
#
# 2. harmonizar Tipo de Vínculo em macrocategorias
#
# 3. tratar valores implausíveis de:
#       idade
#       horas contratuais
#       tempo de emprego
#
# 4. manter Y_doenca exatamente como já validado
#
# 5. excluir da base final:
#       Causa Afastamento
#       Qtd Dias Afastamento
#       Natureza Jurídica bruta
#       CNAE
#       Tipo de Admissão
#       Município
#       CBO de 6 dígitos
#       Grupo_Origem
#       remunerações
#
# 6. produzir auditorias completas antes do ML
#
# NÃO altera nenhuma base anterior.
#
# ============================================================


# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

from google.colab import drive

import os
import re
import glob
from collections import Counter

import numpy as np
import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

drive.mount(
    '/content/drive',
    force_remount=False
)


# ============================================================
# 3. PASTAS
# ============================================================

PASTA_ENTRADA = (
    '/content/drive/MyDrive/TCC_2/dados/'
    'RAIS_HARMONIZADA_ETL'
)


PASTA_SAIDA = (
    '/content/drive/MyDrive/TCC_2/dados/'
    'RAIS_BASE_MODELO_FINAL'
)


PASTA_AUDITORIA = (
    '/content/drive/MyDrive/TCC_2/resultados/'
    'AUDITORIA_FINAL_MODELO'
)


os.makedirs(
    PASTA_SAIDA,
    exist_ok=True
)

os.makedirs(
    PASTA_AUDITORIA,
    exist_ok=True
)


# ============================================================
# 4. CONFIGURAÇÕES
# ============================================================

ANOS = [
    2020,
    2021,
    2022,
    2023,
    2024,
    2025
]


BATCH_SIZE = 250_000


# ------------------------------------------------------------
# Nesta primeira execução:
# True = refazer eventual base final anterior
# ------------------------------------------------------------

SOBRESCREVER = True


# ============================================================
# 5. LIMITES DE CONSISTÊNCIA
# ============================================================

# ------------------------------------------------------------
# Idade
#
# Valores abaixo de 16 anos não fazem sentido para nossa
# população de professores.
#
# Mantemos até 100 anos para não impor corte excessivamente
# restritivo.
# ------------------------------------------------------------

IDADE_MINIMA = 16

IDADE_MAXIMA = 100


# ------------------------------------------------------------
# Horas contratuais
#
# Na base de MODELAGEM:
#
# <= 0  -> ausente
# > 44  -> ausente
#
# O valor original continua preservado nas bases anteriores.
# Aqui apenas impedimos que valores como 60 ou 99 sejam
# interpretados pelo modelo como carga horária real.
# ------------------------------------------------------------

HORAS_MINIMAS = 1

HORAS_MAXIMAS = 44


# ============================================================
# 6. TIPO DE VÍNCULO - HARMONIZAÇÃO
#
# Códigos RAIS:
#
# 10,15,20,25 = CLT / prazo indeterminado
#
# 30,31       = servidor estatutário
#
# 35          = servidor público não efetivo
#
# 95          = determinado - Lei 8.745
# 96          = determinado - Lei Estadual
# 97          = determinado - Lei Municipal
#
# Para reduzir a quebra 2022 -> 2023:
#
# 35,95,96,97
#
# são reunidos na mesma macrocategoria.
#
# ============================================================

MAPA_TIPO_VINCULO_MACRO = {

    # --------------------------------------------------------
    # CLT - prazo indeterminado
    # --------------------------------------------------------

    '10':
        'CLT - prazo indeterminado',

    '15':
        'CLT - prazo indeterminado',

    '20':
        'CLT - prazo indeterminado',

    '25':
        'CLT - prazo indeterminado',


    # --------------------------------------------------------
    # Servidor estatutário
    # --------------------------------------------------------

    '30':
        'Público - estatutário',

    '31':
        'Público - estatutário',


    # --------------------------------------------------------
    # Servidor público não efetivo / temporário
    #
    # Agrupamento especialmente importante para harmonização
    # entre RAIS tradicional e eSocial.
    # --------------------------------------------------------

    '35':
        'Público - não efetivo / temporário',

    '95':
        'Público - não efetivo / temporário',

    '96':
        'Público - não efetivo / temporário',

    '97':
        'Público - não efetivo / temporário',


    # --------------------------------------------------------
    # Trabalhador avulso
    # --------------------------------------------------------

    '40':
        'Avulso',


    # --------------------------------------------------------
    # Temporário / prazo determinado privado
    # --------------------------------------------------------

    '50':
        'Temporário / prazo determinado',

    '60':
        'Temporário / prazo determinado',

    '65':
        'Temporário / prazo determinado',

    '70':
        'Temporário / prazo determinado',

    '75':
        'Temporário / prazo determinado',

    '90':
        'Temporário / prazo determinado',


    # --------------------------------------------------------
    # Aprendiz
    # --------------------------------------------------------

    '55':
        'Aprendiz',


    # --------------------------------------------------------
    # Diretor / dirigente
    # --------------------------------------------------------

    '80':
        'Diretor / dirigente'
}


# ============================================================
# 7. COLUNAS FINAIS
#
# ESTA será a base destinada aos modelos.
# ============================================================

COLUNAS_FINAIS = [

    # --------------------------------------------------------
    # Controle temporal
    #
    # Mantemos para split temporal.
    # Não necessariamente entrará como preditor principal.
    # --------------------------------------------------------

    'Ano',


    # --------------------------------------------------------
    # Geografia
    # --------------------------------------------------------

    'UF',


    # --------------------------------------------------------
    # Ocupação
    # --------------------------------------------------------

    'Familia_CBO',


    # --------------------------------------------------------
    # TARGET
    # --------------------------------------------------------

    'Y_doenca',


    # --------------------------------------------------------
    # Demográficas
    # --------------------------------------------------------

    'Idade',

    'Sexo_codigo',

    'Raca_cor_codigo',

    'Escolaridade_codigo',


    # --------------------------------------------------------
    # Condições do vínculo
    # --------------------------------------------------------

    'Qtd_horas_contratuais',

    'Tempo_emprego_meses',

    'Tipo_vinculo_macro',


    # --------------------------------------------------------
    # Empregador
    # --------------------------------------------------------

    'Natureza_macro',

    'Tamanho_estabelecimento_codigo',


    # --------------------------------------------------------
    # Característica individual
    # --------------------------------------------------------

    'Indicador_deficiencia_codigo'
]


# ============================================================
# 8. PREDITORES DO MODELO PRINCIPAL
#
# Ano fica FORA inicialmente.
#
# Poderemos depois comparar:
#
# modelo sem Ano
# versus
# modelo com Ano
#
# ============================================================

PREDITORES_PRINCIPAIS = [

    'UF',

    'Familia_CBO',

    'Idade',

    'Sexo_codigo',

    'Raca_cor_codigo',

    'Escolaridade_codigo',

    'Qtd_horas_contratuais',

    'Tempo_emprego_meses',

    'Tipo_vinculo_macro',

    'Natureza_macro',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo'
]


TARGET = 'Y_doenca'


# ============================================================
# 9. FUNÇÃO - NORMALIZAR CÓDIGO CATEGÓRICO
#
# Exemplos:
#
# "01"   -> "1"
# "02"   -> "2"
# "09"   -> "9"
# "35.0" -> "35"
#
# ============================================================

def normalizar_codigo_categoria(serie):

    s = (

        serie
        .astype('string')
        .str.strip()
    )


    # --------------------------------------------------------
    # Ausências textuais
    # --------------------------------------------------------

    s = s.mask(

        s.str.upper().isin(
            [
                '',
                'NAN',
                'NONE',
                '<NA>',
                'NULL'
            ]
        )
    )


    # --------------------------------------------------------
    # Converter apenas valores realmente numéricos
    # --------------------------------------------------------

    numero = pd.to_numeric(

        s.str.replace(
            ',',
            '.',
            regex=False
        ),

        errors='coerce'
    )


    mascara_inteiro = (

        numero.notna()

        &

        np.isclose(
            numero,
            numero.round()
        )
    )


    resultado = s.copy()


    if (
        mascara_inteiro.sum()
        >
        0
    ):

        resultado.loc[
            mascara_inteiro
        ] = (

            numero.loc[
                mascara_inteiro
            ]

            .round()

            .astype(
                'Int64'
            )

            .astype(
                'string'
            )
        )


    return resultado.astype(
        'string'
    )


# ============================================================
# 10. NUMÉRICA
# ============================================================

def numerica(serie):

    return pd.to_numeric(

        serie
        .astype('string')
        .str.strip()
        .str.replace(
            ',',
            '.',
            regex=False
        ),

        errors='coerce'
    )


# ============================================================
# 11. IDENTIFICAR ANO E GRUPO
# ============================================================

def identificar_arquivo(caminho):

    nome = os.path.basename(
        caminho
    )


    resultado = re.search(

        r'RAIS_HARMONIZADA_'
        r'(\d{4})_(.+)\.parquet$',

        nome,

        flags=re.IGNORECASE
    )


    if resultado is None:

        raise RuntimeError(

            f'Nome de arquivo não reconhecido: '
            f'{nome}'
        )


    ano = int(
        resultado.group(1)
    )


    grupo = (
        resultado
        .group(2)
        .upper()
    )


    return ano, grupo


# ============================================================
# 12. LOCALIZAR OS 36 PARQUETS
# ============================================================

arquivos = sorted(

    glob.glob(

        os.path.join(
            PASTA_ENTRADA,
            '**',
            '*.parquet'
        ),

        recursive=True
    )
)


print(
    f'Arquivos encontrados: '
    f'{len(arquivos)}'
)


if len(
    arquivos
) != 36:

    print(
        '\nATENÇÃO: eram esperados 36 arquivos.'
    )


# ============================================================
# 13. COLUNAS OBRIGATÓRIAS DE ENTRADA
# ============================================================

COLUNAS_ENTRADA = [

    'Ano',

    'UF',

    'Familia_CBO',

    'Y_doenca',

    'Idade',

    'Sexo_codigo',

    'Raca_cor_codigo',

    'Escolaridade_codigo',

    'Qtd_horas_contratuais',

    'Tempo_emprego',

    'Tipo_vinculo_codigo',

    'Natureza_macro',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo'
]


# ============================================================
# 14. CONTADORES
# ============================================================

status = []


# ------------------------------------------------------------
# Totais de validação
# ------------------------------------------------------------

linhas_entrada_ano = Counter()

linhas_saida_ano = Counter()

target_entrada_ano = Counter()

target_saida_ano = Counter()


# ------------------------------------------------------------
# Outliers
# ------------------------------------------------------------

outliers = Counter()


# ------------------------------------------------------------
# Missing final
# ------------------------------------------------------------

missing_final = Counter()

total_final = Counter()


# ------------------------------------------------------------
# Distribuições categóricas
# ------------------------------------------------------------

distribuicao = Counter()


VARIAVEIS_CATEGORICAS_FINAL = [

    'UF',

    'Familia_CBO',

    'Sexo_codigo',

    'Raca_cor_codigo',

    'Escolaridade_codigo',

    'Tipo_vinculo_macro',

    'Natureza_macro',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo'
]


# ------------------------------------------------------------
# Tipo de vínculo bruto normalizado
# ------------------------------------------------------------

distribuicao_tipo_bruto = Counter()


# ------------------------------------------------------------
# Valores de vínculo não mapeados
# ------------------------------------------------------------

tipo_vinculo_nao_mapeado = Counter()


# ------------------------------------------------------------
# Zeros à esquerda depois da limpeza
# ------------------------------------------------------------

zeros_esquerda = Counter()


# ============================================================
# 15. PROCESSAMENTO
# ============================================================

for indice_arquivo, arquivo in enumerate(

    arquivos,

    start=1
):

    print(
        '\n'
        +
        '=' * 90
    )


    ano, grupo = identificar_arquivo(
        arquivo
    )


    print(

        f'[{indice_arquivo}/'
        f'{len(arquivos)}] '

        f'{ano} - {grupo}'
    )


    parquet = pq.ParquetFile(
        arquivo
    )


    colunas_existentes = set(
        parquet.schema.names
    )


    # ========================================================
    # VALIDAR COLUNAS DE ENTRADA
    # ========================================================

    faltantes = [

        coluna

        for coluna
        in COLUNAS_ENTRADA

        if coluna
        not in colunas_existentes
    ]


    if len(
        faltantes
    ) > 0:

        raise RuntimeError(

            f'Colunas ausentes em '
            f'{os.path.basename(arquivo)}:\n'

            f'{faltantes}'
        )


    # ========================================================
    # SAÍDA
    # ========================================================

    pasta_saida_ano = os.path.join(

        PASTA_SAIDA,

        str(
            ano
        )
    )


    os.makedirs(
        pasta_saida_ano,
        exist_ok=True
    )


    arquivo_saida = os.path.join(

        pasta_saida_ano,

        (
            f'RAIS_MODELO_FINAL_'
            f'{ano}_{grupo}.parquet'
        )
    )


    if (

        SOBRESCREVER

        and

        os.path.exists(
            arquivo_saida
        )
    ):

        os.remove(
            arquivo_saida
        )


    writer = None

    linhas_arquivo = 0

    target_arquivo = 0


    try:

        # ====================================================
        # BATCHES
        # ====================================================

        for numero_batch, batch in enumerate(

            parquet.iter_batches(

                batch_size=
                    BATCH_SIZE,

                columns=
                    COLUNAS_ENTRADA
            ),

            start=1
        ):

            df = batch.to_pandas()


            n = len(
                df
            )


            linhas_arquivo += n

            linhas_entrada_ano[
                ano
            ] += n


            # =================================================
            # ANO
            # =================================================

            ano_serie = pd.Series(

                ano,

                index=df.index,

                dtype='Int16'
            )


            # =================================================
            # UF
            # =================================================

            uf = (

                df[
                    'UF'
                ]

                .astype(
                    'string'
                )

                .str.strip()

                .str.upper()
            )


            # =================================================
            # FAMÍLIA CBO
            #
            # NÃO remover zeros à esquerda genericamente.
            # Apenas garantir 4 dígitos.
            # =================================================

            familia_cbo = (

                df[
                    'Familia_CBO'
                ]

                .astype(
                    'string'
                )

                .str.extract(
                    r'(\d{4})',
                    expand=False
                )
            )


            # =================================================
            # TARGET
            # =================================================

            y = pd.to_numeric(

                df[
                    'Y_doenca'
                ],

                errors='coerce'
            )


            # -------------------------------------------------
            # Segurança
            # -------------------------------------------------

            valores_target = set(

                y
                .dropna()
                .unique()
                .tolist()
            )


            if not valores_target.issubset(
                {
                    0,
                    1
                }
            ):

                raise RuntimeError(

                    f'Y_doenca possui valores '
                    f'inesperados: '

                    f'{valores_target}'
                )


            if y.isna().any():

                raise RuntimeError(

                    'Y_doenca possui valores ausentes.'
                )


            y = (
                y
                .astype(
                    'Int8'
                )
            )


            target_batch = int(
                y.sum()
            )


            target_arquivo += (
                target_batch
            )


            target_entrada_ano[
                ano
            ] += target_batch


            # =================================================
            # IDADE
            # =================================================

            idade_original = numerica(

                df[
                    'Idade'
                ]
            )


            idade_abaixo = (

                idade_original
                <
                IDADE_MINIMA
            )


            idade_acima = (

                idade_original
                >
                IDADE_MAXIMA
            )


            idade_invalida = (

                idade_abaixo.fillna(False)

                |

                idade_acima.fillna(False)
            )


            outliers[
                (
                    ano,
                    'Idade_original_nula'
                )
            ] += int(

                idade_original
                .isna()
                .sum()
            )


            outliers[
                (
                    ano,
                    'Idade_abaixo_16'
                )
            ] += int(

                idade_abaixo
                .fillna(False)
                .sum()
            )


            outliers[
                (
                    ano,
                    'Idade_acima_100'
                )
            ] += int(

                idade_acima
                .fillna(False)
                .sum()
            )


            idade = (

                idade_original

                .mask(
                    idade_invalida
                )

                .astype(
                    'float32'
                )
            )


            # =================================================
            # SEXO
            # =================================================

            sexo = normalizar_codigo_categoria(

                df[
                    'Sexo_codigo'
                ]
            )


            # =================================================
            # RAÇA/COR
            # =================================================

            raca = normalizar_codigo_categoria(

                df[
                    'Raca_cor_codigo'
                ]
            )


            # =================================================
            # ESCOLARIDADE
            # =================================================

            escolaridade = normalizar_codigo_categoria(

                df[
                    'Escolaridade_codigo'
                ]
            )


            # =================================================
            # HORAS CONTRATUAIS
            # =================================================

            horas_original = numerica(

                df[
                    'Qtd_horas_contratuais'
                ]
            )


            horas_nao_positivas = (

                horas_original
                <
                HORAS_MINIMAS
            )


            horas_acima_44 = (

                horas_original
                >
                HORAS_MAXIMAS
            )


            horas_99 = (

                horas_original
                ==
                99
            )


            outliers[
                (
                    ano,
                    'Horas_original_nula'
                )
            ] += int(

                horas_original
                .isna()
                .sum()
            )


            outliers[
                (
                    ano,
                    'Horas_menor_1'
                )
            ] += int(

                horas_nao_positivas
                .fillna(False)
                .sum()
            )


            outliers[
                (
                    ano,
                    'Horas_acima_44'
                )
            ] += int(

                horas_acima_44
                .fillna(False)
                .sum()
            )


            outliers[
                (
                    ano,
                    'Horas_igual_99'
                )
            ] += int(

                horas_99
                .fillna(False)
                .sum()
            )


            horas_invalidas = (

                horas_nao_positivas
                .fillna(False)

                |

                horas_acima_44
                .fillna(False)
            )


            horas = (

                horas_original

                .mask(
                    horas_invalidas
                )

                .astype(
                    'float32'
                )
            )


            # =================================================
            # TEMPO DE EMPREGO
            #
            # A variável RAIS representa meses.
            #
            # 600 meses = 50 anos.
            #
            # Portanto, não tratar 600 como erro.
            # Apenas valores negativos são invalidados.
            # =================================================

            tempo_original = numerica(

                df[
                    'Tempo_emprego'
                ]
            )


            tempo_negativo = (

                tempo_original
                <
                0
            )


            outliers[
                (
                    ano,
                    'Tempo_emprego_original_nulo'
                )
            ] += int(

                tempo_original
                .isna()
                .sum()
            )


            outliers[
                (
                    ano,
                    'Tempo_emprego_negativo'
                )
            ] += int(

                tempo_negativo
                .fillna(False)
                .sum()
            )


            outliers[
                (
                    ano,
                    'Tempo_emprego_600_ou_mais'
                )
            ] += int(

                (
                    tempo_original
                    >=
                    600
                )
                .fillna(False)
                .sum()
            )


            tempo_emprego = (

                tempo_original

                .mask(
                    tempo_negativo
                    .fillna(False)
                )

                .astype(
                    'float32'
                )
            )


            # =================================================
            # TIPO DE VÍNCULO
            # =================================================

            tipo_vinculo_norm = (

                normalizar_codigo_categoria(

                    df[
                        'Tipo_vinculo_codigo'
                    ]
                )
            )


            # -------------------------------------------------
            # Auditoria do código bruto
            # -------------------------------------------------

            contagens_tipo = (

                tipo_vinculo_norm
                .fillna(
                    '<NA>'
                )
                .value_counts()
            )


            for valor, quantidade in (
                contagens_tipo.items()
            ):

                distribuicao_tipo_bruto[
                    (
                        ano,
                        str(valor)
                    )
                ] += int(
                    quantidade
                )


            # -------------------------------------------------
            # Macrocategoria
            # -------------------------------------------------

            tipo_macro = (

                tipo_vinculo_norm

                .map(
                    MAPA_TIPO_VINCULO_MACRO
                )

                .astype(
                    'string'
                )
            )


            # -------------------------------------------------
            # Identificar códigos não mapeados
            # -------------------------------------------------

            mascara_nao_mapeado = (

                tipo_vinculo_norm
                .notna()

                &

                tipo_macro
                .isna()
            )


            if (
                mascara_nao_mapeado
                .any()
            ):

                valores = (

                    tipo_vinculo_norm.loc[
                        mascara_nao_mapeado
                    ]

                    .value_counts()
                )


                for valor, quantidade in (
                    valores.items()
                ):

                    tipo_vinculo_nao_mapeado[
                        (
                            ano,
                            str(valor)
                        )
                    ] += int(
                        quantidade
                    )


            # -------------------------------------------------
            # Não perder observação.
            #
            # Caso apareça código desconhecido:
            # categoria explícita.
            # -------------------------------------------------

            tipo_macro = (

                tipo_macro

                .mask(
                    mascara_nao_mapeado,
                    'Outro / não mapeado'
                )
            )


            # =================================================
            # NATUREZA MACRO
            # =================================================

            natureza_macro = (

                df[
                    'Natureza_macro'
                ]

                .astype(
                    'string'
                )

                .str.strip()
            )


            # =================================================
            # TAMANHO DO ESTABELECIMENTO
            # =================================================

            tamanho = normalizar_codigo_categoria(

                df[
                    'Tamanho_estabelecimento_codigo'
                ]
            )


            # =================================================
            # DEFICIÊNCIA
            # =================================================

            deficiencia = normalizar_codigo_categoria(

                df[
                    'Indicador_deficiencia_codigo'
                ]
            )


            # =================================================
            # BASE FINAL
            # =================================================

            base_final = pd.DataFrame({

                'Ano':
                    ano_serie,

                'UF':
                    uf,

                'Familia_CBO':
                    familia_cbo,

                'Y_doenca':
                    y,

                'Idade':
                    idade,

                'Sexo_codigo':
                    sexo,

                'Raca_cor_codigo':
                    raca,

                'Escolaridade_codigo':
                    escolaridade,

                'Qtd_horas_contratuais':
                    horas,

                'Tempo_emprego_meses':
                    tempo_emprego,

                'Tipo_vinculo_macro':
                    tipo_macro,

                'Natureza_macro':
                    natureza_macro,

                'Tamanho_estabelecimento_codigo':
                    tamanho,

                'Indicador_deficiencia_codigo':
                    deficiencia
            })


            # =================================================
            # GARANTIR ORDEM DAS COLUNAS
            # =================================================

            base_final = (
                base_final[
                    COLUNAS_FINAIS
                ]
            )


            # =================================================
            # AUDITORIA DE ZEROS À ESQUERDA
            # =================================================

            for variavel in [

                'Sexo_codigo',

                'Raca_cor_codigo',

                'Escolaridade_codigo',

                'Tamanho_estabelecimento_codigo',

                'Indicador_deficiencia_codigo'

            ]:

                qtd = (

                    base_final[
                        variavel
                    ]

                    .astype(
                        'string'
                    )

                    .str.match(
                        r'^0\d+$',
                        na=False
                    )

                    .sum()
                )


                zeros_esquerda[
                    (
                        ano,
                        variavel
                    )
                ] += int(
                    qtd
                )


            # =================================================
            # MISSINGNESS FINAL
            # =================================================

            for coluna in (
                COLUNAS_FINAIS
            ):

                total_final[
                    (
                        ano,
                        coluna
                    )
                ] += n


                missing_final[
                    (
                        ano,
                        coluna
                    )
                ] += int(

                    base_final[
                        coluna
                    ]

                    .isna()

                    .sum()
                )


            # =================================================
            # DISTRIBUIÇÃO DAS CATEGÓRICAS
            # =================================================

            for variavel in (
                VARIAVEIS_CATEGORICAS_FINAL
            ):

                contagens = (

                    base_final[
                        variavel
                    ]

                    .astype(
                        'string'
                    )

                    .fillna(
                        '<NA>'
                    )

                    .value_counts(
                        dropna=False
                    )
                )


                for valor, quantidade in (
                    contagens.items()
                ):

                    distribuicao[
                        (
                            ano,
                            variavel,
                            str(valor)
                        )
                    ] += int(
                        quantidade
                    )


            # =================================================
            # SALVAR PARQUET
            # =================================================

            tabela = pa.Table.from_pandas(

                base_final,

                preserve_index=False
            )


            if writer is None:

                writer = pq.ParquetWriter(

                    arquivo_saida,

                    tabela.schema,

                    compression='snappy'
                )


            writer.write_table(
                tabela
            )


            # =================================================
            # VALIDAÇÃO OUTPUT
            # =================================================

            linhas_saida_ano[
                ano
            ] += n


            target_saida_ano[
                ano
            ] += target_batch


            print(

                f'Batch {numero_batch}: '

                f'{linhas_arquivo:,} '
                f'linhas acumuladas | '

                f'Y=1: {target_arquivo:,}'
            )


        # ====================================================
        # FECHAR PARQUET
        # ====================================================

        if writer is not None:

            writer.close()

            writer = None


        # ====================================================
        # VALIDAR ARQUIVO SALVO
        # ====================================================

        pf_saida = pq.ParquetFile(
            arquivo_saida
        )


        linhas_salvas = (
            pf_saida
            .metadata
            .num_rows
        )


        if (
            linhas_salvas
            !=
            linhas_arquivo
        ):

            raise RuntimeError(

                f'Divergência: '
                f'{linhas_arquivo:,} processadas '

                f'e {linhas_salvas:,} salvas.'
            )


        status.append({

            'Ano':
                ano,

            'Grupo':
                grupo,

            'Arquivo_entrada':
                os.path.basename(
                    arquivo
                ),

            'Arquivo_saida':
                os.path.basename(
                    arquivo_saida
                ),

            'Linhas':
                linhas_salvas,

            'Y_doenca_1':
                target_arquivo,

            'Status':
                'OK'
        })


    except Exception as erro:

        if writer is not None:

            writer.close()


        if os.path.exists(
            arquivo_saida
        ):

            try:

                os.remove(
                    arquivo_saida
                )

            except Exception:

                pass


        status.append({

            'Ano':
                ano,

            'Grupo':
                grupo,

            'Arquivo_entrada':
                os.path.basename(
                    arquivo
                ),

            'Arquivo_saida':
                os.path.basename(
                    arquivo_saida
                ),

            'Linhas':
                None,

            'Y_doenca_1':
                None,

            'Status':
                str(
                    erro
                )
        })


        print(
            f'ERRO: {erro}'
        )


# ============================================================
# 16. STATUS
# ============================================================

df_status = pd.DataFrame(
    status
)


df_status.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '01_status_processamento.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 17. VALIDAÇÃO DE LINHAS E TARGET
# ============================================================

registros = []


for ano in ANOS:

    entrada = (
        linhas_entrada_ano[
            ano
        ]
    )


    saida = (
        linhas_saida_ano[
            ano
        ]
    )


    y_entrada = (
        target_entrada_ano[
            ano
        ]
    )


    y_saida = (
        target_saida_ano[
            ano
        ]
    )


    registros.append({

        'Ano':
            ano,

        'Linhas_entrada':
            entrada,

        'Linhas_saida':
            saida,

        'Diferenca_linhas':
            saida
            -
            entrada,

        'Y_entrada':
            y_entrada,

        'Y_saida':
            y_saida,

        'Diferenca_Y':
            y_saida
            -
            y_entrada,

        'Pct_doenca':
            (
                y_saida
                /
                saida
                *
                100
            )
            if saida > 0
            else np.nan
    })


df_validacao = pd.DataFrame(
    registros
)


df_validacao.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '02_validacao_linhas_target.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 18. AUDITORIA DE OUTLIERS
# ============================================================

registros = []


for (
    ano,
    indicador
), quantidade in (
    outliers.items()
):

    n = (
        linhas_entrada_ano[
            ano
        ]
    )


    registros.append({

        'Ano':
            ano,

        'Indicador':
            indicador,

        'Quantidade':
            quantidade,

        'Percentual':
            (
                quantidade
                /
                n
                *
                100
            )
            if n > 0
            else np.nan
    })


df_outliers = pd.DataFrame(
    registros
)


df_outliers = (

    df_outliers

    .sort_values(
        [
            'Indicador',
            'Ano'
        ]
    )

    .reset_index(
        drop=True
    )
)


df_outliers.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '03_outliers_numericos_por_ano.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 19. TIPO DE VÍNCULO BRUTO NORMALIZADO
# ============================================================

registros = []


for (
    ano,
    codigo
), quantidade in (
    distribuicao_tipo_bruto.items()
):

    registros.append({

        'Ano':
            ano,

        'Codigo':
            codigo,

        'Quantidade':
            quantidade
    })


df_tipo_bruto = pd.DataFrame(
    registros
)


if len(
    df_tipo_bruto
) > 0:

    df_tipo_bruto[
        'Percentual'
    ] = (

        df_tipo_bruto[
            'Quantidade'
        ]

        /

        df_tipo_bruto
        .groupby(
            'Ano'
        )[
            'Quantidade'
        ]
        .transform(
            'sum'
        )

        * 100
    )


df_tipo_bruto.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '04_tipo_vinculo_bruto_normalizado.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 20. CÓDIGOS DE VÍNCULO NÃO MAPEADOS
# ============================================================

registros = []


for (
    ano,
    codigo
), quantidade in (
    tipo_vinculo_nao_mapeado.items()
):

    registros.append({

        'Ano':
            ano,

        'Codigo':
            codigo,

        'Quantidade':
            quantidade
    })


df_tipo_nao_mapeado = pd.DataFrame(
    registros
)


df_tipo_nao_mapeado.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '05_tipo_vinculo_nao_mapeado.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 21. MISSINGNESS DA BASE FINAL
# ============================================================

registros = []


for (
    ano,
    variavel
), total in (
    total_final.items()
):

    ausentes = (
        missing_final[
            (
                ano,
                variavel
            )
        ]
    )


    registros.append({

        'Ano':
            ano,

        'Variavel':
            variavel,

        'N':
            total,

        'Ausentes':
            ausentes,

        'Validos':
            total
            -
            ausentes,

        'Pct_ausentes':
            (
                ausentes
                /
                total
                *
                100
            )
            if total > 0
            else np.nan
    })


df_missing = pd.DataFrame(
    registros
)


df_missing = (

    df_missing

    .sort_values(
        [
            'Variavel',
            'Ano'
        ]
    )

    .reset_index(
        drop=True
    )
)


df_missing.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '06_missingness_base_final.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 22. DISTRIBUIÇÕES CATEGÓRICAS
# ============================================================

registros = []


for (
    ano,
    variavel,
    valor
), quantidade in (
    distribuicao.items()
):

    registros.append({

        'Ano':
            ano,

        'Variavel':
            variavel,

        'Valor':
            valor,

        'Quantidade':
            quantidade
    })


df_distribuicao = pd.DataFrame(
    registros
)


if len(
    df_distribuicao
) > 0:

    df_distribuicao[
        'Percentual'
    ] = (

        df_distribuicao[
            'Quantidade'
        ]

        /

        df_distribuicao
        .groupby(
            [
                'Ano',
                'Variavel'
            ]
        )[
            'Quantidade'
        ]
        .transform(
            'sum'
        )

        * 100
    )


    df_distribuicao = (

        df_distribuicao

        .sort_values(
            [
                'Variavel',
                'Ano',
                'Quantidade'
            ],

            ascending=[
                True,
                True,
                False
            ]
        )

        .reset_index(
            drop=True
        )
    )


df_distribuicao.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '07_distribuicao_categorias_final.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 23. ZEROS À ESQUERDA
# ============================================================

registros = []


for (
    ano,
    variavel
), quantidade in (
    zeros_esquerda.items()
):

    registros.append({

        'Ano':
            ano,

        'Variavel':
            variavel,

        'Valores_com_zero_esquerda':
            quantidade
    })


df_zeros = pd.DataFrame(
    registros
)


df_zeros.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '08_validacao_zeros_esquerda.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 24. VALIDAR SCHEMA DOS 36 PARQUETS FINAIS
# ============================================================

arquivos_finais = sorted(

    glob.glob(

        os.path.join(
            PASTA_SAIDA,
            '**',
            '*.parquet'
        ),

        recursive=True
    )
)


schemas = []


for arquivo in (
    arquivos_finais
):

    pf = pq.ParquetFile(
        arquivo
    )


    schema = (
        pf.schema_arrow
    )


    assinatura = ' | '.join(

        [
            f'{campo.name}:{campo.type}'

            for campo
            in schema
        ]
    )


    schemas.append({

        'Arquivo':
            os.path.basename(
                arquivo
            ),

        'Schema':
            assinatura,

        'Qtd_colunas':
            len(
                schema
            ),

        'Linhas':
            pf.metadata.num_rows
    })


df_schema = pd.DataFrame(
    schemas
)


df_schema.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '09_schema_arquivos_finais.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


qtd_schemas = (

    df_schema[
        'Schema'
    ]
    .nunique()

    if len(
        df_schema
    ) > 0

    else 0
)


# ============================================================
# 25. LISTA OFICIAL DE VARIÁVEIS DO MODELO
# ============================================================

registros = []


for variavel in (
    COLUNAS_FINAIS
):

    if variavel == TARGET:

        papel = 'TARGET'

        modelo_principal = True


    elif variavel == 'Ano':

        papel = (
            'Controle temporal / '
            'separação treino-teste'
        )

        modelo_principal = False


    elif variavel in (
        PREDITORES_PRINCIPAIS
    ):

        papel = 'Preditor'

        modelo_principal = True


    else:

        papel = 'Auxiliar'

        modelo_principal = False


    registros.append({

        'Variavel':
            variavel,

        'Papel':
            papel,

        'Modelo_principal':
            modelo_principal
    })


df_preditores = pd.DataFrame(
    registros
)


df_preditores.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '10_preditores_modelo_principal.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 26. VALIDAÇÃO DE LEAKAGE
# ============================================================

TERMOS_PROIBIDOS = [

    'CAUSA',

    'AFAST',

    'DIAS',

    'REMUN',

    'SALARIO',

    'DESLIG',

    'ADMISSAO',

    'MUNICIPIO',

    'CNAE'
]


colunas_suspeitas = []


for coluna in (
    COLUNAS_FINAIS
):

    nome = (
        coluna
        .upper()
    )


    for termo in (
        TERMOS_PROIBIDOS
    ):

        if termo in nome:

            colunas_suspeitas.append(
                coluna
            )


# ============================================================
# 27. RESUMO FINAL DE VALIDAÇÃO
# ============================================================

total_entrada = sum(
    linhas_entrada_ano.values()
)


total_saida = sum(
    linhas_saida_ano.values()
)


total_y_entrada = sum(
    target_entrada_ano.values()
)


total_y_saida = sum(
    target_saida_ano.values()
)


qtd_arquivos_ok = (

    df_status[
        'Status'
    ]
    .eq(
        'OK'
    )
    .sum()
)


total_zeros_esquerda = sum(
    zeros_esquerda.values()
)


total_nao_mapeados = sum(
    tipo_vinculo_nao_mapeado.values()
)


df_resumo_final = pd.DataFrame(

    [

        {
            'Validacao':
                'Arquivos finais',

            'Resultado':
                len(
                    arquivos_finais
                ),

            'Esperado':
                36,

            'OK':
                len(
                    arquivos_finais
                )
                ==
                36
        },

        {
            'Validacao':
                'Arquivos processados OK',

            'Resultado':
                qtd_arquivos_ok,

            'Esperado':
                36,

            'OK':
                qtd_arquivos_ok
                ==
                36
        },

        {
            'Validacao':
                'Linhas entrada = saída',

            'Resultado':
                total_saida,

            'Esperado':
                total_entrada,

            'OK':
                total_saida
                ==
                total_entrada
        },

        {
            'Validacao':
                'Y entrada = Y saída',

            'Resultado':
                total_y_saida,

            'Esperado':
                total_y_entrada,

            'OK':
                total_y_saida
                ==
                total_y_entrada
        },

        {
            'Validacao':
                'Schemas distintos',

            'Resultado':
                qtd_schemas,

            'Esperado':
                1,

            'OK':
                qtd_schemas
                ==
                1
        },

        {
            'Validacao':
                'Zeros à esquerda remanescentes',

            'Resultado':
                total_zeros_esquerda,

            'Esperado':
                0,

            'OK':
                total_zeros_esquerda
                ==
                0
        },

        {
            'Validacao':
                'Variáveis suspeitas de leakage',

            'Resultado':
                str(
                    colunas_suspeitas
                ),

            'Esperado':
                '[]',

            'OK':
                len(
                    colunas_suspeitas
                )
                ==
                0
        },

        {
            'Validacao':
                'Tipo vínculo não mapeado',

            'Resultado':
                total_nao_mapeados,

            'Esperado':
                'Idealmente 0',

            'OK':
                total_nao_mapeados
                ==
                0
        }
    ]
)


df_resumo_final.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '11_validacao_final_resumo.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 28. EXIBIR RESULTADOS
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    'VALIDAÇÃO FINAL'
)

print(
    '=' * 90
)


display(
    df_resumo_final
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'LINHAS E TARGET POR ANO'
)

print(
    '=' * 90
)


display(
    df_validacao
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'OUTLIERS NUMÉRICOS'
)

print(
    '=' * 90
)


display(
    df_outliers
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'MISSINGNESS DA BASE FINAL'
)

print(
    '=' * 90
)


display(
    df_missing
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'TIPO DE VÍNCULO NÃO MAPEADO'
)

print(
    '=' * 90
)


if len(
    df_tipo_nao_mapeado
) == 0:

    print(
        'Nenhum código não mapeado.'
    )

else:

    display(
        df_tipo_nao_mapeado
    )


print(
    '\n'
    +
    '=' * 90
)

print(
    'PREDITORES DO MODELO PRINCIPAL'
)

print(
    '=' * 90
)


display(
    df_preditores
)


print(
    '\n'
    +
    '=' * 90
)

print(
    f'BASE FINAL SALVA EM:\n'
    f'{PASTA_SAIDA}'
)

print(
    '\n'
    +
    f'AUDITORIAS SALVAS EM:\n'
    f'{PASTA_AUDITORIA}'
)

In [ ]:
# ============================================================
# BASE FINAL V2 - CORREÇÕES FINAIS PARA MACHINE LEARNING
# RAIS PROFESSORES - 2020 A 2025
#
# ENTRADA:
#   RAIS_BASE_MODELO_FINAL
#
# SAÍDA:
#   RAIS_BASE_MODELO_FINAL_V2
#
# CORREÇÕES:
#
# 1. MANTER separados no modelo principal:
#
#    Público - estatutário
#    Público - não efetivo / temporário
#
# 2. Criar variável auxiliar para robustez:
#
#    Tipo_vinculo_robustez
#
#    em que os dois grupos públicos são reunidos em "Público"
#
# 3. Transformar:
#
#    "Outro / não mapeado"
#
#    em NA.
#
#    Na auditoria anterior, isso corresponde a apenas
#    4 registros de código 999.
#
# 4. Raça/cor:
#
#    - NÃO entra no modelo principal 2020-2025
#    - é preservada como Raca_cor_codigo_aux
#    - código 99 é convertido em NA
#    - poderá ser utilizada em robustez 2023-2025
#
# 5. Ano:
#
#    permanece na base para controle e separação temporal,
#    mas NÃO entra no modelo principal.
#
# 6. Horas contratuais:
#
#    permanecem como tratadas na etapa anterior.
#    Valores inválidos já estão como NA.
#
# 7. NÃO há exclusão de registros.
#
# ============================================================


# ============================================================
# 1. IMPORTAÇÕES
# ============================================================

from google.colab import drive

import os
import re
import glob

from collections import Counter

import numpy as np
import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq


# ============================================================
# 2. GOOGLE DRIVE
# ============================================================

drive.mount(
    '/content/drive',
    force_remount=False
)


# ============================================================
# 3. PASTAS
# ============================================================

PASTA_ENTRADA = (
    '/content/drive/MyDrive/TCC_2/dados/'
    'RAIS_BASE_MODELO_FINAL'
)


PASTA_SAIDA = (
    '/content/drive/MyDrive/TCC_2/dados/'
    'RAIS_BASE_MODELO_FINAL_V2'
)


PASTA_AUDITORIA = (
    '/content/drive/MyDrive/TCC_2/resultados/'
    'AUDITORIA_FINAL_V2'
)


os.makedirs(
    PASTA_SAIDA,
    exist_ok=True
)


os.makedirs(
    PASTA_AUDITORIA,
    exist_ok=True
)


# ============================================================
# 4. CONFIGURAÇÕES
# ============================================================

ANOS = [
    2020,
    2021,
    2022,
    2023,
    2024,
    2025
]


BATCH_SIZE = 250_000


# ------------------------------------------------------------
# Primeira execução:
# True
#
# Depois que tudo estiver validado, pode mudar para False.
# ------------------------------------------------------------

SOBRESCREVER = True


# ============================================================
# 5. COLUNAS ESPERADAS NA BASE ATUAL
# ============================================================

COLUNAS_ENTRADA = [

    'Ano',

    'UF',

    'Familia_CBO',

    'Y_doenca',

    'Idade',

    'Sexo_codigo',

    'Raca_cor_codigo',

    'Escolaridade_codigo',

    'Qtd_horas_contratuais',

    'Tempo_emprego_meses',

    'Tipo_vinculo_macro',

    'Natureza_macro',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo'
]


# ============================================================
# 6. COLUNAS DA BASE V2
# ============================================================

COLUNAS_SAIDA = [

    # --------------------------------------------------------
    # Controle temporal
    # --------------------------------------------------------

    'Ano',

    # --------------------------------------------------------
    # Geografia
    # --------------------------------------------------------

    'UF',

    # --------------------------------------------------------
    # Ocupação
    # --------------------------------------------------------

    'Familia_CBO',

    # --------------------------------------------------------
    # TARGET
    # --------------------------------------------------------

    'Y_doenca',

    # --------------------------------------------------------
    # Demográficas
    # --------------------------------------------------------

    'Idade',

    'Sexo_codigo',

    # --------------------------------------------------------
    # Raça permanece SOMENTE como auxiliar.
    # Não utilizar no modelo principal 2020-2025.
    # --------------------------------------------------------

    'Raca_cor_codigo_aux',

    'Escolaridade_codigo',

    # --------------------------------------------------------
    # Condições de trabalho
    # --------------------------------------------------------

    'Qtd_horas_contratuais',

    'Tempo_emprego_meses',

    # --------------------------------------------------------
    # Tipo de vínculo PRINCIPAL:
    #
    # estatutário e temporário público separados.
    # --------------------------------------------------------

    'Tipo_vinculo_macro',

    # --------------------------------------------------------
    # Tipo de vínculo para ROBUSTEZ:
    #
    # todos os públicos agregados.
    # --------------------------------------------------------

    'Tipo_vinculo_robustez',

    # --------------------------------------------------------
    # Natureza jurídica harmonizada
    # --------------------------------------------------------

    'Natureza_macro',

    # --------------------------------------------------------
    # Estabelecimento
    # --------------------------------------------------------

    'Tamanho_estabelecimento_codigo',

    # --------------------------------------------------------
    # Deficiência
    # --------------------------------------------------------

    'Indicador_deficiencia_codigo'
]


# ============================================================
# 7. PREDITORES DO MODELO PRINCIPAL
#
# IMPORTANTE:
#
# Não entram:
#
# - Ano
# - Raça/cor
# - Tipo_vinculo_robustez
#
# ============================================================

PREDITORES_PRINCIPAIS = [

    'UF',

    'Familia_CBO',

    'Idade',

    'Sexo_codigo',

    'Escolaridade_codigo',

    'Qtd_horas_contratuais',

    'Tempo_emprego_meses',

    'Tipo_vinculo_macro',

    'Natureza_macro',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo'
]


TARGET = 'Y_doenca'


# ============================================================
# 8. PREDITORES - ROBUSTEZ 1
#
# Junta:
#
# Público estatutário
# +
# Público não efetivo / temporário
#
# ============================================================

PREDITORES_ROBUSTEZ_VINCULO_AGREGADO = [

    'UF',

    'Familia_CBO',

    'Idade',

    'Sexo_codigo',

    'Escolaridade_codigo',

    'Qtd_horas_contratuais',

    'Tempo_emprego_meses',

    'Tipo_vinculo_robustez',

    'Natureza_macro',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo'
]


# ============================================================
# 9. PREDITORES - ROBUSTEZ 2
#
# Retira completamente Tipo de Vínculo.
# ============================================================

PREDITORES_ROBUSTEZ_SEM_VINCULO = [

    'UF',

    'Familia_CBO',

    'Idade',

    'Sexo_codigo',

    'Escolaridade_codigo',

    'Qtd_horas_contratuais',

    'Tempo_emprego_meses',

    'Natureza_macro',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo'
]


# ============================================================
# 10. PREDITORES - ROBUSTEZ 3
#
# Modelo principal SEM horas contratuais.
#
# Útil caso horas apareça com importância excessiva no SHAP.
# ============================================================

PREDITORES_ROBUSTEZ_SEM_HORAS = [

    'UF',

    'Familia_CBO',

    'Idade',

    'Sexo_codigo',

    'Escolaridade_codigo',

    'Tempo_emprego_meses',

    'Tipo_vinculo_macro',

    'Natureza_macro',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo'
]


# ============================================================
# 11. PREDITORES - ROBUSTEZ 4
#
# Apenas 2023-2025:
#
# acrescenta raça/cor.
# ============================================================

PREDITORES_ROBUSTEZ_2023_2025_COM_RACA = [

    'UF',

    'Familia_CBO',

    'Idade',

    'Sexo_codigo',

    'Raca_cor_codigo_aux',

    'Escolaridade_codigo',

    'Qtd_horas_contratuais',

    'Tempo_emprego_meses',

    'Tipo_vinculo_macro',

    'Natureza_macro',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo'
]


# ============================================================
# 12. TIPOS DE VÍNCULO ESPERADOS
# ============================================================

TIPOS_VINCULO_PRINCIPAL = {

    'CLT - prazo indeterminado',

    'Público - estatutário',

    'Público - não efetivo / temporário',

    'Avulso',

    'Temporário / prazo determinado',

    'Aprendiz',

    'Diretor / dirigente'
}


# ============================================================
# 13. PADRONIZAR STRING
# ============================================================

def padronizar_string(serie):

    s = (

        serie
        .astype('string')
        .str.strip()
    )


    s = s.mask(

        s.str.upper().isin(
            [
                '',
                'NAN',
                'NONE',
                '<NA>',
                'NULL'
            ]
        )
    )


    return s.astype(
        'string'
    )


# ============================================================
# 14. NORMALIZAR CÓDIGO CATEGÓRICO
# ============================================================

def normalizar_codigo(serie):

    s = padronizar_string(
        serie
    )


    numero = pd.to_numeric(

        s.str.replace(
            ',',
            '.',
            regex=False
        ),

        errors='coerce'
    )


    inteiro = (

        numero.notna()

        &

        np.isclose(
            numero,
            numero.round()
        )
    )


    resultado = s.copy()


    resultado.loc[
        inteiro
    ] = (

        numero.loc[
            inteiro
        ]

        .round()

        .astype(
            'Int64'
        )

        .astype(
            'string'
        )
    )


    return resultado.astype(
        'string'
    )


# ============================================================
# 15. IDENTIFICAR ARQUIVO
# ============================================================

def identificar_arquivo(caminho):

    nome = os.path.basename(
        caminho
    )


    resultado = re.search(

        r'RAIS_MODELO_FINAL_'
        r'(\d{4})_(.+)\.parquet$',

        nome,

        flags=re.IGNORECASE
    )


    if resultado is None:

        raise RuntimeError(

            f'Nome não reconhecido: '
            f'{nome}'
        )


    ano = int(
        resultado.group(1)
    )


    grupo = (

        resultado
        .group(2)
        .upper()
    )


    return ano, grupo


# ============================================================
# 16. LOCALIZAR PARQUETS
# ============================================================

arquivos = sorted(

    glob.glob(

        os.path.join(
            PASTA_ENTRADA,
            '**',
            '*.parquet'
        ),

        recursive=True
    )
)


print(
    f'Arquivos encontrados: '
    f'{len(arquivos)}'
)


if len(
    arquivos
) != 36:

    print(
        '\nATENÇÃO: eram esperados '
        '36 arquivos.'
    )


# ============================================================
# 17. CONTADORES
# ============================================================

status = []


# ------------------------------------------------------------
# Entrada x saída
# ------------------------------------------------------------

linhas_entrada = Counter()

linhas_saida = Counter()

target_entrada = Counter()

target_saida = Counter()


# ------------------------------------------------------------
# Vínculo
# ------------------------------------------------------------

vinculo_principal = Counter()

vinculo_robustez = Counter()

vinculo_na = Counter()

vinculo_outro = Counter()


# ------------------------------------------------------------
# Raça
# ------------------------------------------------------------

raca_original = Counter()

raca_aux = Counter()

raca_99_convertida = Counter()


# ------------------------------------------------------------
# Missing
# ------------------------------------------------------------

missing = Counter()

total_variavel = Counter()


# ------------------------------------------------------------
# Distribuições principais
# ------------------------------------------------------------

distribuicoes = Counter()


VARIAVEIS_CATEGORICAS = [

    'UF',

    'Familia_CBO',

    'Sexo_codigo',

    'Raca_cor_codigo_aux',

    'Escolaridade_codigo',

    'Tipo_vinculo_macro',

    'Tipo_vinculo_robustez',

    'Natureza_macro',

    'Tamanho_estabelecimento_codigo',

    'Indicador_deficiencia_codigo'
]


# ============================================================
# 18. PROCESSAR ARQUIVOS
# ============================================================

for indice, arquivo in enumerate(

    arquivos,

    start=1
):

    print(
        '\n'
        +
        '=' * 90
    )


    ano, grupo = identificar_arquivo(
        arquivo
    )


    print(

        f'[{indice}/{len(arquivos)}] '
        f'{ano} - {grupo}'
    )


    parquet = pq.ParquetFile(
        arquivo
    )


    colunas = set(
        parquet.schema.names
    )


    # ========================================================
    # VALIDAR SCHEMA DE ENTRADA
    # ========================================================

    faltantes = [

        coluna

        for coluna
        in COLUNAS_ENTRADA

        if coluna
        not in colunas
    ]


    if len(
        faltantes
    ) > 0:

        raise RuntimeError(

            f'Colunas ausentes em '
            f'{os.path.basename(arquivo)}:\n'

            f'{faltantes}'
        )


    # ========================================================
    # ARQUIVO DE SAÍDA
    # ========================================================

    pasta_ano = os.path.join(

        PASTA_SAIDA,

        str(
            ano
        )
    )


    os.makedirs(
        pasta_ano,
        exist_ok=True
    )


    arquivo_saida = os.path.join(

        pasta_ano,

        (
            f'RAIS_MODELO_FINAL_V2_'
            f'{ano}_{grupo}.parquet'
        )
    )


    if (

        SOBRESCREVER

        and

        os.path.exists(
            arquivo_saida
        )
    ):

        os.remove(
            arquivo_saida
        )


    writer = None

    linhas_arquivo = 0

    y_arquivo = 0


    try:

        # ====================================================
        # BATCHES
        # ====================================================

        for numero_batch, batch in enumerate(

            parquet.iter_batches(

                batch_size=
                    BATCH_SIZE,

                columns=
                    COLUNAS_ENTRADA
            ),

            start=1
        ):

            df = batch.to_pandas()


            n = len(
                df
            )


            linhas_arquivo += n

            linhas_entrada[
                ano
            ] += n


            # =================================================
            # ANO
            # =================================================

            ano_serie = pd.Series(

                ano,

                index=df.index,

                dtype='Int16'
            )


            # =================================================
            # UF
            # =================================================

            uf = (

                df[
                    'UF'
                ]

                .astype(
                    'string'
                )

                .str.strip()

                .str.upper()
            )


            # =================================================
            # FAMÍLIA CBO
            # =================================================

            familia = (

                df[
                    'Familia_CBO'
                ]

                .astype(
                    'string'
                )

                .str.extract(
                    r'(\d{4})',
                    expand=False
                )
            )


            # =================================================
            # TARGET
            # =================================================

            y = pd.to_numeric(

                df[
                    'Y_doenca'
                ],

                errors='coerce'
            )


            if y.isna().any():

                raise RuntimeError(

                    'Y_doenca possui '
                    'valores ausentes.'
                )


            valores_y = set(

                y
                .unique()
                .tolist()
            )


            if not valores_y.issubset(
                {
                    0,
                    1
                }
            ):

                raise RuntimeError(

                    f'Y_doenca inválido: '
                    f'{valores_y}'
                )


            y = y.astype(
                'Int8'
            )


            y_batch = int(
                y.sum()
            )


            y_arquivo += y_batch


            target_entrada[
                ano
            ] += y_batch


            # =================================================
            # IDADE
            # =================================================

            idade = pd.to_numeric(

                df[
                    'Idade'
                ],

                errors='coerce'
            ).astype(
                'float32'
            )


            # =================================================
            # SEXO
            # =================================================

            sexo = normalizar_codigo(

                df[
                    'Sexo_codigo'
                ]
            )


            # =================================================
            # RAÇA/COR
            #
            # Preservar para robustez.
            #
            # Código 99 -> NA.
            #
            # Não usar no modelo principal 2020-2025.
            # =================================================

            raca_original_norm = normalizar_codigo(

                df[
                    'Raca_cor_codigo'
                ]
            )


            contagens_raca_original = (

                raca_original_norm

                .fillna(
                    '<NA>'
                )

                .value_counts()
            )


            for valor, quantidade in (
                contagens_raca_original.items()
            ):

                raca_original[
                    (
                        ano,
                        str(valor)
                    )
                ] += int(
                    quantidade
                )


            mascara_raca_99 = (

                raca_original_norm
                ==
                '99'
            )


            raca_99_convertida[
                ano
            ] += int(

                mascara_raca_99
                .fillna(False)
                .sum()
            )


            raca_aux = (

                raca_original_norm

                .mask(
                    mascara_raca_99,
                    pd.NA
                )

                .astype(
                    'string'
                )
            )


            # =================================================
            # ESCOLARIDADE
            # =================================================

            escolaridade = normalizar_codigo(

                df[
                    'Escolaridade_codigo'
                ]
            )


            # =================================================
            # HORAS
            #
            # Já tratadas na base anterior.
            # Apenas preservar.
            # =================================================

            horas = pd.to_numeric(

                df[
                    'Qtd_horas_contratuais'
                ],

                errors='coerce'
            ).astype(
                'float32'
            )


            # =================================================
            # TEMPO DE EMPREGO
            # =================================================

            tempo = pd.to_numeric(

                df[
                    'Tempo_emprego_meses'
                ],

                errors='coerce'
            ).astype(
                'float32'
            )


            # =================================================
            # TIPO DE VÍNCULO PRINCIPAL
            #
            # NÃO unir:
            #
            # Público - estatutário
            #
            # com
            #
            # Público - não efetivo / temporário
            # =================================================

            tipo_principal = padronizar_string(

                df[
                    'Tipo_vinculo_macro'
                ]
            )


            # -------------------------------------------------
            # Os 4 casos identificados anteriormente.
            #
            # Não transformar em categoria substantiva.
            # -------------------------------------------------

            mascara_outro = (

                tipo_principal
                .str.upper()
                ==
                'OUTRO / NÃO MAPEADO'
            )


            vinculo_outro[
                ano
            ] += int(

                mascara_outro
                .fillna(False)
                .sum()
            )


            tipo_principal = (

                tipo_principal

                .mask(
                    mascara_outro,
                    pd.NA
                )

                .astype(
                    'string'
                )
            )


            # -------------------------------------------------
            # Validar categorias conhecidas
            # -------------------------------------------------

            categorias_encontradas = set(

                tipo_principal
                .dropna()
                .unique()
                .tolist()
            )


            categorias_inesperadas = (

                categorias_encontradas

                -

                TIPOS_VINCULO_PRINCIPAL
            )


            if len(
                categorias_inesperadas
            ) > 0:

                raise RuntimeError(

                    'Categorias inesperadas '
                    'em Tipo_vinculo_macro: '

                    f'{categorias_inesperadas}'
                )


            # -------------------------------------------------
            # Auditoria
            # -------------------------------------------------

            contagens = (

                tipo_principal

                .fillna(
                    '<NA>'
                )

                .value_counts()
            )


            for valor, quantidade in (
                contagens.items()
            ):

                vinculo_principal[
                    (
                        ano,
                        str(valor)
                    )
                ] += int(
                    quantidade
                )


            vinculo_na[
                ano
            ] += int(

                tipo_principal
                .isna()
                .sum()
            )


            # =================================================
            # TIPO DE VÍNCULO PARA ROBUSTEZ
            #
            # Aqui SIM unimos os dois grupos públicos.
            #
            # Esta variável NÃO entra no modelo principal.
            # =================================================

            tipo_robustez = (

                tipo_principal.copy()
            )


            mascara_publico = (

                tipo_robustez.isin(
                    [
                        'Público - estatutário',

                        (
                            'Público - não efetivo '
                            '/ temporário'
                        )
                    ]
                )
            )


            tipo_robustez.loc[
                mascara_publico
            ] = 'Público'


            tipo_robustez = (
                tipo_robustez
                .astype(
                    'string'
                )
            )


            contagens = (

                tipo_robustez

                .fillna(
                    '<NA>'
                )

                .value_counts()
            )


            for valor, quantidade in (
                contagens.items()
            ):

                vinculo_robustez[
                    (
                        ano,
                        str(valor)
                    )
                ] += int(
                    quantidade
                )


            # =================================================
            # NATUREZA JURÍDICA
            # =================================================

            natureza = padronizar_string(

                df[
                    'Natureza_macro'
                ]
            )


            # =================================================
            # TAMANHO DO ESTABELECIMENTO
            # =================================================

            tamanho = normalizar_codigo(

                df[
                    'Tamanho_estabelecimento_codigo'
                ]
            )


            # =================================================
            # DEFICIÊNCIA
            # =================================================

            deficiencia = normalizar_codigo(

                df[
                    'Indicador_deficiencia_codigo'
                ]
            )


            # =================================================
            # BASE FINAL V2
            # =================================================

            base = pd.DataFrame({

                'Ano':
                    ano_serie,

                'UF':
                    uf,

                'Familia_CBO':
                    familia,

                'Y_doenca':
                    y,

                'Idade':
                    idade,

                'Sexo_codigo':
                    sexo,

                'Raca_cor_codigo_aux':
                    raca_aux,

                'Escolaridade_codigo':
                    escolaridade,

                'Qtd_horas_contratuais':
                    horas,

                'Tempo_emprego_meses':
                    tempo,

                'Tipo_vinculo_macro':
                    tipo_principal,

                'Tipo_vinculo_robustez':
                    tipo_robustez,

                'Natureza_macro':
                    natureza,

                'Tamanho_estabelecimento_codigo':
                    tamanho,

                'Indicador_deficiencia_codigo':
                    deficiencia
            })


            base = (
                base[
                    COLUNAS_SAIDA
                ]
            )


            # =================================================
            # MISSINGNESS
            # =================================================

            for coluna in (
                COLUNAS_SAIDA
            ):

                total_variavel[
                    (
                        ano,
                        coluna
                    )
                ] += n


                missing[
                    (
                        ano,
                        coluna
                    )
                ] += int(

                    base[
                        coluna
                    ]
                    .isna()
                    .sum()
                )


            # =================================================
            # DISTRIBUIÇÕES CATEGÓRICAS
            # =================================================

            for variavel in (
                VARIAVEIS_CATEGORICAS
            ):

                contagens = (

                    base[
                        variavel
                    ]

                    .astype(
                        'string'
                    )

                    .fillna(
                        '<NA>'
                    )

                    .value_counts()
                )


                for valor, quantidade in (
                    contagens.items()
                ):

                    distribuicoes[
                        (
                            ano,
                            variavel,
                            str(valor)
                        )
                    ] += int(
                        quantidade
                    )


            # =================================================
            # SALVAR
            # =================================================

            tabela = pa.Table.from_pandas(

                base,

                preserve_index=False
            )


            if writer is None:

                writer = pq.ParquetWriter(

                    arquivo_saida,

                    tabela.schema,

                    compression='snappy'
                )


            writer.write_table(
                tabela
            )


            # =================================================
            # CONTADORES DE SAÍDA
            # =================================================

            linhas_saida[
                ano
            ] += n


            target_saida[
                ano
            ] += y_batch


            print(

                f'Batch {numero_batch}: '

                f'{linhas_arquivo:,} linhas | '

                f'Y=1: {y_arquivo:,}'
            )


        # ====================================================
        # FECHAR PARQUET
        # ====================================================

        if writer is not None:

            writer.close()

            writer = None


        # ====================================================
        # VALIDAR ARQUIVO
        # ====================================================

        pf_saida = pq.ParquetFile(
            arquivo_saida
        )


        linhas_salvas = (

            pf_saida
            .metadata
            .num_rows
        )


        if (
            linhas_salvas
            !=
            linhas_arquivo
        ):

            raise RuntimeError(

                f'Divergência: '
                f'{linhas_arquivo:,} processadas '

                f'e {linhas_salvas:,} salvas.'
            )


        status.append({

            'Ano':
                ano,

            'Grupo':
                grupo,

            'Arquivo_entrada':
                os.path.basename(
                    arquivo
                ),

            'Arquivo_saida':
                os.path.basename(
                    arquivo_saida
                ),

            'Linhas':
                linhas_salvas,

            'Y_doenca_1':
                y_arquivo,

            'Status':
                'OK'
        })


    except Exception as erro:

        if writer is not None:

            writer.close()


        if os.path.exists(
            arquivo_saida
        ):

            try:

                os.remove(
                    arquivo_saida
                )

            except Exception:

                pass


        status.append({

            'Ano':
                ano,

            'Grupo':
                grupo,

            'Arquivo_entrada':
                os.path.basename(
                    arquivo
                ),

            'Arquivo_saida':
                os.path.basename(
                    arquivo_saida
                ),

            'Linhas':
                None,

            'Y_doenca_1':
                None,

            'Status':
                str(
                    erro
                )
        })


        print(
            f'ERRO: {erro}'
        )


# ============================================================
# 19. STATUS
# ============================================================

df_status = pd.DataFrame(
    status
)


df_status.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '01_status_processamento.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 20. VALIDAÇÃO DE LINHAS E TARGET
# ============================================================

registros = []


for ano in ANOS:

    registros.append({

        'Ano':
            ano,

        'Linhas_entrada':
            linhas_entrada[
                ano
            ],

        'Linhas_saida':
            linhas_saida[
                ano
            ],

        'Diferenca_linhas':
            (
                linhas_saida[
                    ano
                ]
                -
                linhas_entrada[
                    ano
                ]
            ),

        'Y_entrada':
            target_entrada[
                ano
            ],

        'Y_saida':
            target_saida[
                ano
            ],

        'Diferenca_Y':
            (
                target_saida[
                    ano
                ]
                -
                target_entrada[
                    ano
                ]
            ),

        'Pct_doenca':
            (
                target_saida[
                    ano
                ]
                /
                linhas_saida[
                    ano
                ]
                *
                100
            )
            if linhas_saida[
                ano
            ] > 0
            else np.nan
    })


df_validacao = pd.DataFrame(
    registros
)


df_validacao.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '02_validacao_linhas_target.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 21. DISTRIBUIÇÃO DO VÍNCULO PRINCIPAL
# ============================================================

registros = []


for (
    ano,
    categoria
), quantidade in (
    vinculo_principal.items()
):

    registros.append({

        'Ano':
            ano,

        'Tipo_vinculo_macro':
            categoria,

        'Quantidade':
            quantidade
    })


df_vinculo_principal = pd.DataFrame(
    registros
)


if len(
    df_vinculo_principal
) > 0:

    df_vinculo_principal[
        'Percentual'
    ] = (

        df_vinculo_principal[
            'Quantidade'
        ]

        /

        df_vinculo_principal
        .groupby(
            'Ano'
        )[
            'Quantidade'
        ]
        .transform(
            'sum'
        )

        * 100
    )


    df_vinculo_principal = (

        df_vinculo_principal

        .sort_values(
            [
                'Ano',
                'Quantidade'
            ],

            ascending=[
                True,
                False
            ]
        )
    )


df_vinculo_principal.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '03_tipo_vinculo_principal.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 22. DISTRIBUIÇÃO DO VÍNCULO DE ROBUSTEZ
# ============================================================

registros = []


for (
    ano,
    categoria
), quantidade in (
    vinculo_robustez.items()
):

    registros.append({

        'Ano':
            ano,

        'Tipo_vinculo_robustez':
            categoria,

        'Quantidade':
            quantidade
    })


df_vinculo_robustez = pd.DataFrame(
    registros
)


if len(
    df_vinculo_robustez
) > 0:

    df_vinculo_robustez[
        'Percentual'
    ] = (

        df_vinculo_robustez[
            'Quantidade'
        ]

        /

        df_vinculo_robustez
        .groupby(
            'Ano'
        )[
            'Quantidade'
        ]
        .transform(
            'sum'
        )

        * 100
    )


    df_vinculo_robustez = (

        df_vinculo_robustez

        .sort_values(
            [
                'Ano',
                'Quantidade'
            ],

            ascending=[
                True,
                False
            ]
        )
    )


df_vinculo_robustez.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '04_tipo_vinculo_robustez.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 23. AUDITORIA DA RAÇA/COR
# ============================================================

registros = []


for (
    ano,
    codigo
), quantidade in (
    raca_original.items()
):

    registros.append({

        'Ano':
            ano,

        'Codigo_original':
            codigo,

        'Quantidade':
            quantidade
    })


df_raca_original = pd.DataFrame(
    registros
)


if len(
    df_raca_original
) > 0:

    df_raca_original[
        'Percentual'
    ] = (

        df_raca_original[
            'Quantidade'
        ]

        /

        df_raca_original
        .groupby(
            'Ano'
        )[
            'Quantidade'
        ]
        .transform(
            'sum'
        )

        * 100
    )


df_raca_original.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '05_raca_cor_original_por_ano.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 24. CÓDIGO 99 DE RAÇA TRANSFORMADO EM NA
# ============================================================

df_raca_99 = pd.DataFrame(

    [

        {

            'Ano':
                ano,

            'Codigo_99_convertido_NA':
                raca_99_convertida[
                    ano
                ],

            'Percentual_da_base':
                (
                    raca_99_convertida[
                        ano
                    ]
                    /
                    linhas_saida[
                        ano
                    ]
                    *
                    100
                )
                if linhas_saida[
                    ano
                ] > 0
                else np.nan
        }

        for ano in ANOS
    ]
)


df_raca_99.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '06_raca_99_convertida_na.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 25. MISSINGNESS FINAL
# ============================================================

registros = []


for (
    ano,
    variavel
), total in (
    total_variavel.items()
):

    ausentes = missing[
        (
            ano,
            variavel
        )
    ]


    registros.append({

        'Ano':
            ano,

        'Variavel':
            variavel,

        'N':
            total,

        'Ausentes':
            ausentes,

        'Validos':
            total
            -
            ausentes,

        'Pct_ausentes':
            (
                ausentes
                /
                total
                *
                100
            )
            if total > 0
            else np.nan
    })


df_missing = pd.DataFrame(
    registros
)


df_missing = (

    df_missing

    .sort_values(
        [
            'Variavel',
            'Ano'
        ]
    )

    .reset_index(
        drop=True
    )
)


df_missing.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '07_missingness_base_v2.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 26. DISTRIBUIÇÕES CATEGÓRICAS
# ============================================================

registros = []


for (
    ano,
    variavel,
    valor
), quantidade in (
    distribuicoes.items()
):

    registros.append({

        'Ano':
            ano,

        'Variavel':
            variavel,

        'Valor':
            valor,

        'Quantidade':
            quantidade
    })


df_distribuicoes = pd.DataFrame(
    registros
)


if len(
    df_distribuicoes
) > 0:

    df_distribuicoes[
        'Percentual'
    ] = (

        df_distribuicoes[
            'Quantidade'
        ]

        /

        df_distribuicoes
        .groupby(
            [
                'Ano',
                'Variavel'
            ]
        )[
            'Quantidade'
        ]
        .transform(
            'sum'
        )

        * 100
    )


df_distribuicoes.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '08_distribuicoes_categoricas_v2.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 27. CONFIGURAÇÕES DOS MODELOS
# ============================================================

configuracoes = []


def adicionar_configuracao(
    nome,
    anos,
    preditores,
    observacao
):

    for ordem, variavel in enumerate(

        preditores,

        start=1
    ):

        configuracoes.append({

            'Modelo':
                nome,

            'Anos':
                anos,

            'Ordem':
                ordem,

            'Variavel':
                variavel,

            'Target':
                TARGET,

            'Observacao':
                observacao
        })


# ------------------------------------------------------------
# A - MODELO PRINCIPAL
# ------------------------------------------------------------

adicionar_configuracao(

    'A_PRINCIPAL',

    '2020-2025',

    PREDITORES_PRINCIPAIS,

    (
        'Estatutários e temporários públicos '
        'mantidos separados. '
        'Sem raça/cor e sem Ano como preditor.'
    )
)


# ------------------------------------------------------------
# B - ROBUSTEZ COM PÚBLICO AGREGADO
# ------------------------------------------------------------

adicionar_configuracao(

    'B_PUBLICO_AGREGADO',

    '2020-2025',

    PREDITORES_ROBUSTEZ_VINCULO_AGREGADO,

    (
        'Estatutários e temporários públicos '
        'agregados em Público.'
    )
)


# ------------------------------------------------------------
# C - ROBUSTEZ SEM TIPO DE VÍNCULO
# ------------------------------------------------------------

adicionar_configuracao(

    'C_SEM_TIPO_VINCULO',

    '2020-2025',

    PREDITORES_ROBUSTEZ_SEM_VINCULO,

    (
        'Remove totalmente Tipo de Vínculo.'
    )
)


# ------------------------------------------------------------
# D - ROBUSTEZ SEM HORAS
# ------------------------------------------------------------

adicionar_configuracao(

    'D_SEM_HORAS',

    '2020-2025',

    PREDITORES_ROBUSTEZ_SEM_HORAS,

    (
        'Remove horas contratuais para avaliar '
        'possível efeito da ruptura 2022-2023.'
    )
)


# ------------------------------------------------------------
# E - ROBUSTEZ 2023-2025 COM RAÇA/COR
# ------------------------------------------------------------

adicionar_configuracao(

    'E_2023_2025_COM_RACA',

    '2023-2025',

    PREDITORES_ROBUSTEZ_2023_2025_COM_RACA,

    (
        'Inclui raça/cor apenas no período '
        'com codificação mais comparável.'
    )
)


df_configuracoes = pd.DataFrame(
    configuracoes
)


df_configuracoes.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '09_configuracoes_modelagem.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 28. VALIDAR SCHEMA DOS 36 ARQUIVOS
# ============================================================

arquivos_v2 = sorted(

    glob.glob(

        os.path.join(
            PASTA_SAIDA,
            '**',
            '*.parquet'
        ),

        recursive=True
    )
)


schemas = []


for arquivo in (
    arquivos_v2
):

    pf = pq.ParquetFile(
        arquivo
    )


    schema = (
        pf.schema_arrow
    )


    assinatura = ' | '.join(

        [

            f'{campo.name}:'
            f'{campo.type}'

            for campo
            in schema
        ]
    )


    schemas.append({

        'Arquivo':
            os.path.basename(
                arquivo
            ),

        'Linhas':
            pf.metadata.num_rows,

        'Qtd_colunas':
            len(
                schema
            ),

        'Schema':
            assinatura
    })


df_schema = pd.DataFrame(
    schemas
)


df_schema.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '10_schema_base_v2.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


qtd_schemas = (

    df_schema[
        'Schema'
    ]
    .nunique()

    if len(
        df_schema
    ) > 0

    else 0
)


# ============================================================
# 29. VALIDAÇÕES FINAIS
# ============================================================

total_entrada = sum(
    linhas_entrada.values()
)


total_saida = sum(
    linhas_saida.values()
)


total_y_entrada = sum(
    target_entrada.values()
)


total_y_saida = sum(
    target_saida.values()
)


total_vinculo_outro = sum(
    vinculo_outro.values()
)


total_vinculo_na = sum(
    vinculo_na.values()
)


arquivos_ok = (

    df_status[
        'Status'
    ]
    .eq(
        'OK'
    )
    .sum()
)


# ------------------------------------------------------------
# Verificar se ainda existe "Outro / não mapeado"
# ------------------------------------------------------------

outro_remanescente = 0


for (
    ano,
    categoria
), quantidade in (
    vinculo_principal.items()
):

    if (
        categoria
        ==
        'Outro / não mapeado'
    ):

        outro_remanescente += (
            quantidade
        )


# ============================================================
# 30. RESUMO FINAL
# ============================================================

df_resumo_final = pd.DataFrame(

    [

        {
            'Validacao':
                'Arquivos V2',

            'Resultado':
                len(
                    arquivos_v2
                ),

            'Esperado':
                36,

            'OK':
                len(
                    arquivos_v2
                )
                ==
                36
        },

        {
            'Validacao':
                'Arquivos processados OK',

            'Resultado':
                arquivos_ok,

            'Esperado':
                36,

            'OK':
                arquivos_ok
                ==
                36
        },

        {
            'Validacao':
                'Linhas preservadas',

            'Resultado':
                total_saida,

            'Esperado':
                total_entrada,

            'OK':
                total_saida
                ==
                total_entrada
        },

        {
            'Validacao':
                'Y preservado',

            'Resultado':
                total_y_saida,

            'Esperado':
                total_y_entrada,

            'OK':
                total_y_saida
                ==
                total_y_entrada
        },

        {
            'Validacao':
                'Schemas distintos',

            'Resultado':
                qtd_schemas,

            'Esperado':
                1,

            'OK':
                qtd_schemas
                ==
                1
        },

        {
            'Validacao':
                'Outro / não mapeado remanescente',

            'Resultado':
                outro_remanescente,

            'Esperado':
                0,

            'OK':
                outro_remanescente
                ==
                0
        },

        {
            'Validacao':
                'Registros convertidos de Outro para NA',

            'Resultado':
                total_vinculo_outro,

            'Esperado':
                4,

            'OK':
                total_vinculo_outro
                ==
                4
        },

        {
            'Validacao':
                'NA em Tipo de Vínculo após correção',

            'Resultado':
                total_vinculo_na,

            'Esperado':
                4,

            'OK':
                total_vinculo_na
                ==
                4
        }
    ]
)


df_resumo_final.to_csv(

    os.path.join(
        PASTA_AUDITORIA,
        '11_validacao_final_v2.csv'
    ),

    index=False,

    encoding='utf-8-sig'
)


# ============================================================
# 31. EXIBIR RESULTADOS
# ============================================================

print(
    '\n'
    +
    '=' * 90
)

print(
    'VALIDAÇÃO FINAL V2'
)

print(
    '=' * 90
)


display(
    df_resumo_final
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'LINHAS E TARGET POR ANO'
)

print(
    '=' * 90
)


display(
    df_validacao
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'TIPO DE VÍNCULO - MODELO PRINCIPAL'
)

print(
    '=' * 90
)


display(
    df_vinculo_principal
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'TIPO DE VÍNCULO - ROBUSTEZ COM PÚBLICO AGREGADO'
)

print(
    '=' * 90
)


display(
    df_vinculo_robustez
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'RAÇA/COR - CÓDIGO 99 TRANSFORMADO EM NA'
)

print(
    '=' * 90
)


display(
    df_raca_99
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'MISSINGNESS DA BASE V2'
)

print(
    '=' * 90
)


display(
    df_missing
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'CONFIGURAÇÕES PREVISTAS PARA MODELAGEM'
)

print(
    '=' * 90
)


display(
    df_configuracoes
)


print(
    '\n'
    +
    '=' * 90
)

print(
    'BASE V2 SALVA EM'
)

print(
    '=' * 90
)


print(
    PASTA_SAIDA
)


print(
    '\nAUDITORIAS SALVAS EM:'
)

print(
    PASTA_AUDITORIA
)